# Heart Disease Classification Analysis - Colab Pipeline
This notebook contains the full Heart Disease classification pipeline.

### ⚠️ Instructions
1. Please **upload `heart.csv`** to the default `/content` directory in this Colab environment before running the cells below.
2. Run all cells sequentially. The first cell will install any dependencies. The next few cells will write the python scripts into the Colab environment, and the subsequent cells will execute the pipeline.

In [1]:
!pip install shap imbalanced-learn scikit-learn pandas numpy matplotlib seaborn joblib

In [2]:
%%writefile phase1_foundation.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Create directories for outputs
os.makedirs('eda', exist_ok=True)
os.makedirs('splits', exist_ok=True)

# Step 1: Load data
print("Step 1: Loading data...")
df = pd.read_csv('heart.csv')
print(f"Data shape: {df.shape}")

# Step 2: Data audit
print("\nStep 2: Data audit...")
print("\nColumns and data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nBasic statistics:")
print(df.describe(include='all'))

# Step 3: Target variable analysis
print("\nStep 3: Target variable analysis...")
target_col = 'target'  # Assuming the target column is named 'target'
if target_col not in df.columns:
    # Try to find target column
    possible_targets = ['target', 'heart_disease', 'diagnosis', 'class']
    for col in possible_targets:
        if col in df.columns:
            target_col = col
            break
    else:
        target_col = df.columns[-1]  # Default to last column
        print(f"No obvious target column found, using '{target_col}' as target")

print(f"Target column: '{target_col}'")
print(f"Target distribution:\n{df[target_col].value_counts()}")
print(f"Target distribution (%):\n{df[target_col].value_counts(normalize=True) * 100}")

# Step 4: EDA - Univariate analysis
print("\nStep 4: EDA - Univariate analysis...")
# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Identify feature types
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Plot numeric features distribution
if numeric_cols:
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(numeric_cols, 1):
        plt.subplot(4, 4, i)
        sns.histplot(data=X, x=col, kde=True)
        plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.savefig('eda/numeric_features_distribution.png')
    plt.close()

# Plot categorical features distribution
if categorical_cols:
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(categorical_cols, 1):
        plt.subplot(3, 3, i)
        sns.countplot(data=X, x=col)
        plt.title(f'Distribution of {col}')
        plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('eda/categorical_features_distribution.png')
    plt.close()

# Step 5: EDA - Bivariate analysis with target
print("\nStep 5: EDA - Bivariate analysis with target...")
# Numeric features vs target
if numeric_cols:
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(numeric_cols, 1):
        plt.subplot(4, 4, i)
        sns.boxplot(data=df, x=target_col, y=col)
        plt.title(f'{col} vs {target_col}')
    plt.tight_layout()
    plt.savefig('eda/numeric_vs_target.png')
    plt.close()

# Categorical features vs target
if categorical_cols:
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(categorical_cols, 1):
        plt.subplot(3, 3, i)
        sns.countplot(data=df, x=col, hue=target_col)
        plt.title(f'{col} vs {target_col}')
        plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('eda/categorical_vs_target.png')
    plt.close()

# Step 6: Correlation analysis
print("\nStep 6: Correlation analysis...")
plt.figure(figsize=(12, 10))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('eda/correlation_matrix.png')
plt.close()

# Check for high correlation with target (potential leakage indicators)
target_correlations = correlation_matrix[target_col].abs().sort_values(ascending=False)
print(f"\nTop 5 features correlated with target:\n{target_correlations.head()}")

# Step 7: Data leakage check
print("\nStep 7: Data leakage check...")
# Check for any feature that is almost perfectly correlated with target (could be leakage)
high_corr_features = target_correlations[target_correlations > 0.9].index.tolist()
high_corr_features = [f for f in high_corr_features if f != target_col]
if high_corr_features:
    print(f"WARNING: Features with >0.9 correlation with target (potential leakage): {high_corr_features}")
else:
    print("No features with >0.9 correlation with target found.")

# Check for identifier-like columns
id_like_cols = [col for col in df.columns if 'id' in col.lower() or 'index' in col.lower()]
if id_like_cols:
    print(f"Identifier-like columns found: {id_like_cols}")
    print("Consider removing these from features if they are not predictive.")
else:
    print("No obvious identifier-like columns found.")

# Step 8: Train/Validation/Test split
print("\nStep 8: Train/Validation/Test split...")
# First split: train+val vs test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: train vs val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val  # 0.25 * 0.8 = 0.2
)

print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Validation set size: {X_val.shape[0]} samples ({X_val.shape[0]/len(df)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.1f}%)")

# Save splits
train_df = pd.concat([X_train, y_train], axis=1)
val_df = pd.concat([X_val, y_val], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv('splits/train.csv', index=False)
val_df.to_csv('splits/val.csv', index=False)
test_df.to_csv('splits/test.csv', index=False)

print("\nSplits saved to 'splits/' directory:")
print("- train.csv")
print("- val.csv")
print("- test.csv")

print("\nEDA plots saved to 'eda/' directory.")

print("\nPhase 1 (Steps 1-7) completed successfully!!")



Writing phase1_foundation.py


In [3]:
%%writefile phase2_modeling_foundation.py
# Phase 2: Modeling foundation: preprocessing, feature engineering, baselines, CV

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif

warnings.filterwarnings('ignore')

# Create directories for outputs
os.makedirs('models', exist_ok=True)
os.makedirs('eda', exist_ok=True)  # for any additional plots

print('Phase 2: Modeling foundation')
print('='*50)

# Load the splits
print('\n1. Loading train, validation, and test splits...')
train_df = pd.read_csv('splits/train.csv')
val_df = pd.read_csv('splits/val.csv')
test_df = pd.read_csv('splits/test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Validation shape: {val_df.shape}')
print(f'Test shape: {test_df.shape}')

# Separate features and target
target_col = 'target'
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_val = val_df.drop(columns=[target_col])
y_val = val_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f'\nFeature shapes:')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}, y_val: {y_val.shape}')
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

# Identify feature types
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Since all columns are int64 or float64, we need to decide which are categorical
# Based on earlier analysis: sex, cp, fbs, restecg, exang, slope, thal are categorical (though encoded as int)
# We'll define categorical columns based on domain knowledge
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
# Ensure they exist in the dataframe
categorical_cols = [col for col in categorical_cols if col in X_train.columns]
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

print(f'\nNumeric features ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # though no missing values
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Fit preprocessor on training data
print('\n2. Fitting preprocessor on training data...')
preprocessor.fit(X_train)

# Transform the data
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f'Processed feature shapes:')
print(f'X_train_processed: {X_train_processed.shape}')
print(f'X_val_processed: {X_val_processed.shape}')
print(f'X_test_processed: {X_test_processed.shape}')

# Feature engineering: Polynomial features for numeric components?
# We'll add polynomial features (degree=2) for the numeric part after scaling?
# But note: after one-hot encoding, we have many features. We'll apply polynomial only to the original numeric features.
# We'll create a separate pipeline for polynomial features.
from sklearn.preprocessing import PolynomialFeatures

# Create a transformer that applies polynomial features to numeric features
poly = PolynomialFeatures(degree=2, include_bias=False)
# We'll fit on the numeric training data (after scaling? Usually polynomial features are applied before scaling?
# Common practice: scale then polynomial? Actually, polynomial features can be sensitive to scale, so we should scale first.
# But we already scaled in the numeric_transformer. However, the ColumnTransformer outputs the scaled numeric features.
# We'll extract the numeric part after preprocessing? That's messy.
# Instead, let's create a preprocessing pipeline that includes polynomial features as an option.
# For simplicity, we'll skip polynomial features for now and just use the original features.
# We can add them later if needed.
print('\n3. Skipping polynomial feature engineering for baseline models.')

# We'll use the processed data as is for baselines.
X_train_final = X_train_processed
X_val_final = X_val_processed
X_test_final = X_test_processed

# Save the preprocessor and processed data for later use
print('\n3.5. Saving preprocessor and processed data...')
preprocessor_path = 'models/preprocessor.pkl'
joblib.dump(preprocessor, preprocessor_path)
print(f'Preprocessor saved to {preprocessor_path}')

# Optionally save the processed data
train_processed_df = pd.DataFrame(X_train_processed, index=X_train.index)
train_processed_df[target_col] = y_train.values
train_processed_df.to_csv('splits/train_processed.csv', index=False)

val_processed_df = pd.DataFrame(X_val_processed, index=X_val.index)
val_processed_df[target_col] = y_val.values
val_processed_df.to_csv('splits/val_processed.csv', index=False)

test_processed_df = pd.DataFrame(X_test_processed, index=X_test.index)
test_processed_df[target_col] = y_test.values
test_processed_df.to_csv('splits/test_processed.csv', index=False)

print('Processed data saved to splits/ directory.')

# Baseline models
print('\n4. Training baseline models...')
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100)
}

# Train and evaluate each model
results = []
for name, model in models.items():
    print(f'\nTraining {name}...')
    model.fit(X_train_final, y_train)

    # Predict on validation set
    y_val_pred = model.predict(X_val_final)
    y_val_pred_proba = model.predict_proba(X_val_final)[:, 1] if hasattr(model, 'predict_proba') else None

    # Calculate metrics
    accuracy = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)
    roc_auc = roc_auc_score(y_val, y_val_pred_proba) if y_val_pred_proba is not None else None

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'ROC_AUC': roc_auc
    })

    print(f'Validation Accuracy: {accuracy:.4f}')
    print(f'Validation Precision: {precision:.4f}')
    print(f'Validation Recall: {recall:.4f}')
    print(f'Validation F1: {f1:.4f}')
    if roc_auc:
        print(f'Validation ROC-AUC: {roc_auc:.4f}')

    # Save the model
    model_path = f'models/{name.replace(' ', '_').lower()}_baseline.pkl'
    joblib.dump(model, model_path)
    print(f'Model saved to {model_path}')

# Cross-validation on training set
print('\n5. Performing cross-validation on training set...')
cv_results = []
for name, model in models.items():
    print(f'\nCross-validating {name}...')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_final, y_train, cv=cv, scoring='accuracy')
    cv_results.append({
        'Model': name,
        'CV Accuracy Mean': cv_scores.mean(),
        'CV Accuracy Std': cv_scores.std()
    })
    print(f'CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')

# Save results to a CSV file
results_df = pd.DataFrame(results)
cv_df = pd.DataFrame(cv_results)

results_df.to_csv('models/baseline_validation_results.csv', index=False)
cv_df.to_csv('models/baseline_cv_results.csv', index=False)

print('\n6. Baseline results saved to models/ directory.')
print('\nPhase 2 completed successfully!')

Writing phase2_modeling_foundation.py


In [4]:
%%writefile phase3_advanced.py
# Phase 3 (Steps 13-16): imbalance, tuning, stability, and model shortlist.
# The final test split is never read by this script. CV contains all learned operations.
# Outputs are written to models/phase3.

from __future__ import annotations
import json
import warnings
from pathlib import Path
from typing import Any, Literal, TypedDict, cast
import numpy as np
import pandas as pd
from numpy.typing import ArrayLike, NDArray
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
warnings.filterwarnings("ignore")

SEED = 42
CV_FOLDS = 5
STABILITY_SEEDS = (11, 22, 33, 44, 55)
ROOT = Path(__file__).resolve().parent
OUTPUT_DIR = ROOT / "models" / "phase3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class ModelConfig(TypedDict):
    strategy: str
    params: dict[str, Any]

ModelName = Literal["Logistic Regression", "KNN", "SVM", "Random Forest"]


def load_training_data() -> tuple[pd.DataFrame, pd.Series]:
    # Load only the Phase 1 training split; validation/test remain untouched.
    train = pd.read_csv(ROOT / "splits" / "train.csv")
    if "target" not in train.columns:
        raise ValueError("splits/train.csv must contain a 'target' column")
    return train.drop(columns=["target"]), train["target"].astype(int)


def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    categorical = [c for c in ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"] if c in X]
    numeric = [c for c in X.columns if c not in categorical]
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        [("numeric", numeric_pipe, numeric), ("categorical", categorical_pipe, categorical)],
        remainder="drop",
    )


def model_specs() -> dict[ModelName, BaseEstimator]:
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
        "KNN": KNeighborsClassifier(),
        "SVM": SVC(probability=True, random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=250, random_state=SEED, n_jobs=-1),
    }


def build_pipeline(
    X: pd.DataFrame,
    model_name: ModelName,
    strategy: str,
    params: dict[str, Any] | None = None,
) -> ImbPipeline:
    model: Any = clone(model_specs()[model_name])
    if params:
        model_params: dict[str, Any] = {
            key.removeprefix("model__"): value for key, value in params.items()
        }
        model.set_params(**model_params)
    if strategy == "class_weight" and model_name in {"Logistic Regression", "SVM", "Random Forest"}:
        model.set_params(class_weight="balanced")
    preprocessor: Any = make_preprocessor(X)
    model_step: Any = model
    sampler: Any = SMOTE(random_state=SEED, k_neighbors=3)
    if strategy == "smote":
        steps: Any = [
            ("preprocess", preprocessor),
            ("smote", sampler),
            ("model", model_step),
        ]
    else:
        steps: Any = [("preprocess", preprocessor), ("model", model_step)]
    return ImbPipeline(steps)


def scoring() -> dict[str, str]:
    return {"roc_auc": "roc_auc", "pr_auc": "average_precision", "f1": "f1", "recall": "recall"}


def safe_mean(values: ArrayLike) -> float:
    return float(np.nanmean(np.asarray(values, dtype=float)))


def imbalance_experiments(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
    strategies_by_model: dict[ModelName, tuple[str, ...]] = {
        "Logistic Regression": ("none", "class_weight", "smote"),
        "KNN": ("none", "smote"),
        "SVM": ("none", "class_weight", "smote"),
        "Random Forest": ("none", "class_weight", "smote"),
    }
    for model_name, strategies in strategies_by_model.items():
        for strategy in strategies:
            pipeline: Any = build_pipeline(X, model_name, strategy)
            result = cast(dict[str, NDArray[np.float64]], cross_validate(
                pipeline, X, y, cv=cv, scoring=scoring(), n_jobs=-1, error_score="raise",
            ))
            test_roc_auc = np.asarray(result["test_roc_auc"], dtype=float)
            test_pr_auc = np.asarray(result["test_pr_auc"], dtype=float)
            test_f1 = np.asarray(result["test_f1"], dtype=float)
            test_recall = np.asarray(result["test_recall"], dtype=float)
            rows.append({
                "model": model_name,
                "strategy": strategy,
                "roc_auc_mean": safe_mean(test_roc_auc),
                "roc_auc_std": float(np.std(test_roc_auc, ddof=1)),
                "pr_auc_mean": safe_mean(test_pr_auc),
                "f1_mean": safe_mean(test_f1),
                "recall_mean": safe_mean(test_recall),
            })
    frame = pd.DataFrame(rows).sort_values(["model", "roc_auc_mean"], ascending=[True, False])
    frame.to_csv(OUTPUT_DIR / "imbalance_comparison.csv", index=False)
    best = frame.groupby("model", as_index=False).first()
    best.to_csv(OUTPUT_DIR / "selected_imbalance_strategy.csv", index=False)
    return frame

def tuning(
    X: pd.DataFrame,
    y: pd.Series,
    imbalance: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[ModelName, ModelConfig]]:
    search_spaces: dict[ModelName, dict[str, list[Any]]] = {
        "Logistic Regression": {"model__C": [0.1, 1.0, 10.0]},
        "KNN": {"model__n_neighbors": [3, 5, 7, 11], "model__weights": ["uniform", "distance"]},
        "SVM": {"model__C": [0.1, 1.0, 10.0], "model__gamma": ["scale", "auto"]},
        "Random Forest": {
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2"],
        },
    }
    rows: list[dict[str, Any]] = []
    best_configs: dict[ModelName, ModelConfig] = {}
    cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
    for model_name, grid in search_spaces.items():
        strategy = str(
            imbalance.loc[imbalance["model"] == model_name]
            .sort_values("roc_auc_mean", ascending=False)
            .iloc[0]["strategy"]
        )
        search: Any = GridSearchCV(
            build_pipeline(X, model_name, strategy), grid, scoring="roc_auc",
            cv=cv, n_jobs=-1, refit=True, return_train_score=False, error_score="raise",
        )
        search.fit(X, y)
        raw_params = cast(list[Any], cast(dict[str, Any], search.cv_results_)["params"])
        candidate_params: list[dict[str, Any]] = [cast(dict[str, Any], item) for item in raw_params]
        best_params: dict[str, Any] = dict(cast(dict[str, Any], search.best_params_))
        rows.append({
            "model": model_name,
            "strategy": strategy,
            "best_cv_roc_auc": float(search.best_score_),
            "best_params": json.dumps(best_params, sort_keys=True),
            "evaluated_candidates": len(candidate_params),
        })
        best_configs[model_name] = ModelConfig(strategy=strategy, params=best_params)
    frame = pd.DataFrame(rows).sort_values("best_cv_roc_auc", ascending=False)
    frame.to_csv(OUTPUT_DIR / "tuning_results.csv", index=False)
    with open(OUTPUT_DIR / "best_configurations.json", "w", encoding="utf-8") as handle:
        json.dump(best_configs, handle, indent=2)
    return frame, best_configs

def metrics(y_true: ArrayLike, probabilities: ArrayLike) -> dict[str, float]:
    labels: NDArray[np.int_] = np.asarray(y_true, dtype=int)
    scores: NDArray[np.float64] = np.asarray(probabilities, dtype=float)
    predictions: NDArray[np.int_] = np.asarray(scores >= 0.5, dtype=int)
    negatives: NDArray[np.bool_] = np.asarray(labels == 0, dtype=bool)
    true_negatives = int(np.count_nonzero((predictions == 0) & negatives))
    negative_total = int(np.count_nonzero(negatives))
    specificity = true_negatives / negative_total if negative_total > 0 else 0.0
    return {
        "roc_auc": float(roc_auc_score(labels, scores)),
        "pr_auc": float(average_precision_score(labels, scores)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "specificity": specificity,
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "brier": float(brier_score_loss(labels, scores)),
    }


def stability(
    X: pd.DataFrame,
    y: pd.Series,
    configs: dict[ModelName, ModelConfig],
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for model_name, config in configs.items():
        for seed in STABILITY_SEEDS:
            cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=seed)
            estimator = build_pipeline(X, model_name, config["strategy"], config["params"])
            result: dict[str, NDArray[np.float64]] = cast(
                dict[str, NDArray[np.float64]],
                cross_validate(estimator, X, y, cv=cv, scoring=scoring(), n_jobs=-1, error_score="raise"),
            )
            test_roc_auc = np.asarray(result["test_roc_auc"], dtype=float)
            test_pr_auc = np.asarray(result["test_pr_auc"], dtype=float)
            test_f1 = np.asarray(result["test_f1"], dtype=float)
            test_recall = np.asarray(result["test_recall"], dtype=float)
            rows.append({
                "model": model_name,
                "seed": seed,
                "roc_auc_mean": safe_mean(test_roc_auc),
                "roc_auc_std_folds": float(np.std(test_roc_auc, ddof=1)),
                "pr_auc_mean": safe_mean(test_pr_auc),
                "f1_mean": safe_mean(test_f1),
                "recall_mean": safe_mean(test_recall),
            })
    detail = pd.DataFrame(rows)
    detail.to_csv(OUTPUT_DIR / "stability_by_seed.csv", index=False)
    summary = detail.groupby("model", as_index=False).agg(
        roc_auc_mean=("roc_auc_mean", "mean"), roc_auc_std=("roc_auc_mean", "std"),
        pr_auc_mean=("pr_auc_mean", "mean"), f1_mean=("f1_mean", "mean"),
        recall_mean=("recall_mean", "mean"),
    )
    summary["stability_score"] = summary["roc_auc_mean"] - summary["roc_auc_std"].fillna(0)
    summary.sort_values("stability_score", ascending=False).to_csv(OUTPUT_DIR / "stability_summary.csv", index=False)
    return summary

def shortlist(
    tuned: pd.DataFrame,
    stable: pd.DataFrame,
) -> pd.DataFrame:
    leaderboard = tuned.merge(stable, on="model", how="left")
    leaderboard["shortlist_rank"] = leaderboard["stability_score"].rank(method="min", ascending=False).astype(int)
    leaderboard["selected_for_phase4"] = leaderboard["shortlist_rank"] <= 3
    leaderboard = leaderboard.sort_values(["selected_for_phase4", "stability_score"], ascending=[False, False])
    leaderboard.to_csv(OUTPUT_DIR / "model_shortlist_leaderboard.csv", index=False)
    selected = leaderboard.loc[leaderboard["selected_for_phase4"], "model"].tolist()
    with open(OUTPUT_DIR / "phase3_decision.json", "w", encoding="utf-8") as handle:
        json.dump({
            "primary_metric": "ROC-AUC",
            "secondary_metrics": ["PR-AUC", "F1", "recall", "specificity", "precision", "Brier score"],
            "test_set_used": False,
            "shortlisted_models": selected,
            "selection_rule": "top three by stability score = mean ROC-AUC minus across-seed standard deviation",
            "phase4_note": "Use these CV-selected candidates for ensemble research; do not use final test results for selection.",
        }, handle, indent=2)
    return leaderboard

def main() -> None:
    print("Phase 3: advanced modeling (steps 13-16)")
    X, y = load_training_data()
    print(f"Training rows: {len(X)}; features: {X.shape[1]}; positive rate: {y.mean():.3f}")
    imbalance = imbalance_experiments(X, y)
    tuned, configs = tuning(X, y, imbalance)
    stable = stability(X, y, configs)
    board = shortlist(tuned, stable)
    print("\nPhase 3 completed successfully.")
    print(board[["model", "best_cv_roc_auc", "roc_auc_mean", "roc_auc_std", "selected_for_phase4"]].to_string(index=False))
    print(f"Outputs: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()


Writing phase3_advanced.py


In [5]:
%%writefile phase4_ensemble_ablation.py
# Phase 4 (Steps 17–18): Research centerpiece: KNN+SVM+RF hybrid ensemble + ablation
# Strict anti-leakage protocol: test split (splits/test.csv) is never accessed or imported.
# All probability generation, weight optimization, ablation, and stability analysis
# occur strictly within Out-Of-Fold (OOF) cross-validation on splits/train.csv.
# Outputs are written to models/phase4/.

from __future__ import annotations
import json
import sys
import warnings
from pathlib import Path
from typing import Any, Literal, cast

# Ensure stdout uses UTF-8 or safe encoding
if hasattr(sys.stdout, "reconfigure"):
    try:
        getattr(sys.stdout, "reconfigure")(encoding="utf-8")
    except Exception:
        pass

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from numpy.typing import ArrayLike, NDArray
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# Configuration and Reproducibility Setup
# -----------------------------------------------------------------------------
SEED = 42
CV_FOLDS = 5
STABILITY_SEEDS = (11, 22, 33, 44, 55)

ROOT = Path(__file__).resolve().parent
OUTPUT_DIR = ROOT / "models" / "phase4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ModelName = Literal["KNN", "SVM", "Random Forest"]


# -----------------------------------------------------------------------------
# Data Loading & Preprocessing
# -----------------------------------------------------------------------------
def load_training_and_val_data() -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    Loads training and validation splits.
    CRITICAL: splits/test.csv is explicitly NOT loaded to prevent data leakage.
    """
    train = pd.read_csv(ROOT / "splits" / "train.csv")
    val = pd.read_csv(ROOT / "splits" / "val.csv")
    if "target" not in train.columns or "target" not in val.columns:
        raise ValueError("Both train.csv and val.csv must contain a 'target' column")

    X_train = train.drop(columns=["target"])
    y_train = train["target"].astype(int)
    X_val = val.drop(columns=["target"])
    y_val = val["target"].astype(int)
    return X_train, y_train, X_val, y_val


def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """
    Constructs the canonical ColumnTransformer identical to Phase 2/Phase 3.
    """
    categorical = [c for c in ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"] if c in X]
    numeric = [c for c in X.columns if c not in categorical]
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        [("numeric", numeric_pipe, numeric), ("categorical", categorical_pipe, categorical)],
        remainder="drop",
    )


def load_shortlisted_configs() -> dict[str, Any]:
    """
    Loads best configurations found in Phase 3.
    """
    config_path = ROOT / "models" / "phase3" / "best_configurations.json"
    if not config_path.exists():
        raise FileNotFoundError(f"Missing Phase 3 configuration at {config_path}")
    with open(config_path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_pipeline(
    X: pd.DataFrame,
    model_name: ModelName,
    best_configs: dict[str, Any],
) -> ImbPipeline:
    """
    Builds the leakage-safe pipeline for each component model with tuned parameters.
    """
    cfg = best_configs[model_name]
    strategy = cfg["strategy"]
    raw_params = cfg["params"]

    if model_name == "KNN":
        base_model = KNeighborsClassifier()
    elif model_name == "SVM":
        base_model = SVC(probability=True, random_state=SEED)
    elif model_name == "Random Forest":
        base_model = RandomForestClassifier(n_estimators=250, random_state=SEED, n_jobs=-1)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    # Set hyperparameters
    model_params = {k.removeprefix("model__"): v for k, v in raw_params.items()}
    base_model.set_params(**model_params)

    if strategy == "class_weight" and hasattr(base_model, "class_weight"):
        base_model.set_params(class_weight="balanced")

    preprocessor = make_preprocessor(X)

    if strategy == "smote":
        sampler = SMOTE(random_state=SEED, k_neighbors=3)
        return ImbPipeline([
            ("preprocess", preprocessor),
            ("smote", sampler),
            ("model", base_model),
        ])
    else:
        return ImbPipeline([
            ("preprocess", preprocessor),
            ("model", base_model),
        ])


# -----------------------------------------------------------------------------
# Metric Hierarchy Computation
# -----------------------------------------------------------------------------
def calculate_metrics(y_true: ArrayLike, probabilities: ArrayLike, threshold: float = 0.5) -> dict[str, float]:
    labels = np.asarray(y_true, dtype=int)
    scores = np.asarray(probabilities, dtype=float)
    predictions = (scores >= threshold).astype(int)

    negatives = (labels == 0)
    true_negatives = int(np.count_nonzero((predictions == 0) & negatives))
    negative_total = int(np.count_nonzero(negatives))
    specificity = true_negatives / negative_total if negative_total > 0 else 0.0

    return {
        "roc_auc": float(roc_auc_score(labels, scores)),
        "pr_auc": float(average_precision_score(labels, scores)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "specificity": float(specificity),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "brier": float(brier_score_loss(labels, scores)),
    }


# -----------------------------------------------------------------------------
# Scikit-Learn Compatible Hybrid Ensemble Classifier
# -----------------------------------------------------------------------------
class HybridEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Authoritative Hybrid Ensemble combining KNN, SVM, and Random Forest pipelines
    via soft probability voting with configurable weights.
    """
    def __init__(
        self,
        weights: dict[str, float] | None = None,
        best_configs: dict[str, Any] | None = None,
        threshold: float = 0.5,
    ):
        self.weights = weights or {"KNN": 1.0 / 3.0, "SVM": 1.0 / 3.0, "Random Forest": 1.0 / 3.0}
        self.best_configs = best_configs
        self.threshold = threshold
        self.models_: dict[str, ImbPipeline] = {}
        self.classes_: NDArray[np.int_] = np.array([0, 1])

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "HybridEnsembleClassifier":
        configs = self.best_configs or load_shortlisted_configs()
        self.models_ = {
            name: build_pipeline(X, cast(ModelName, name), configs).fit(X, y)
            for name in ["KNN", "SVM", "Random Forest"]
        }
        self.classes_ = np.unique(y)
        return self

    def predict_proba(self, X: pd.DataFrame) -> NDArray[np.float64]:
        total_w = sum(self.weights.values())
        norm_weights = {k: v / total_w for k, v in self.weights.items()}
        combined_prob = np.zeros(len(X), dtype=float)
        for name, model in self.models_.items():
            proba = model.predict_proba(X)[:, 1]
            combined_prob += norm_weights[name] * proba
        return np.column_stack([1.0 - combined_prob, combined_prob])

    def predict(self, X: pd.DataFrame) -> NDArray[np.int_]:
        prob_pos = self.predict_proba(X)[:, 1]
        return (prob_pos >= self.threshold).astype(int)


# -----------------------------------------------------------------------------
# Step 17: Hybrid Ensemble Construction & Weight Optimization
# -----------------------------------------------------------------------------
def generate_oof_predictions(
    X: pd.DataFrame,
    y: pd.Series,
    best_configs: dict[str, Any],
    cv_seed: int = SEED,
) -> tuple[dict[str, NDArray[np.float64]], dict[str, list[dict[str, float]]]]:
    """
    Generates Out-of-Fold (OOF) positive-class probabilities using Stratified K-Fold CV.
    Returns:
      - oof_probs: dict mapping model name to 1D array of OOF probabilities of length len(X).
      - fold_metrics: dict mapping model name to list of metric dicts for each fold.
    """
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=cv_seed)
    model_names: list[ModelName] = ["KNN", "SVM", "Random Forest"]
    oof_probs: dict[str, NDArray[np.float64]] = {
        name: np.zeros(len(X), dtype=float) for name in model_names
    }
    fold_metrics: dict[str, list[dict[str, float]]] = {
        name: [] for name in model_names
    }

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

        for name in model_names:
            pipeline = build_pipeline(X_tr, name, best_configs)
            pipeline.fit(X_tr, y_tr)
            pred_probs = pipeline.predict_proba(X_va)[:, 1]
            oof_probs[name][val_idx] = pred_probs
            fold_metrics[name].append(calculate_metrics(y_va, pred_probs))

    return oof_probs, fold_metrics


def optimize_ensemble_weights(
    y_true: NDArray[np.int_],
    oof_probs: dict[str, NDArray[np.float64]],
    models: list[str],
) -> dict[str, float]:
    """
    Finds optimal weights (w_i >= 0, sum w_i = 1) strictly on training OOF predictions
    by minimizing Brier score loss (strictly proper scoring rule).
    """
    P = np.column_stack([oof_probs[m] for m in models])
    k = len(models)

    def loss(weights: NDArray[np.float64]) -> float:
        w = weights / np.sum(weights)
        p_ens = np.dot(P, w)
        return float(brier_score_loss(y_true, p_ens))

    init_weights = np.ones(k) / k
    bounds = [(0.0, 1.0) for _ in range(k)]
    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}

    res = minimize(
        loss,
        init_weights,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-9},
    )

    raw_w = np.maximum(0.0, res.x)
    norm_w = raw_w / np.sum(raw_w)
    return {model: float(norm_w[i]) for i, model in enumerate(models)}


# -----------------------------------------------------------------------------
# Step 18: Ensemble Ablation Study & Diversity Analysis
# -----------------------------------------------------------------------------
def run_ablation_study(
    y_true: NDArray[np.int_],
    oof_probs: dict[str, NDArray[np.float64]],
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Evaluates all 7 combinations of KNN, SVM, and Random Forest:
      1. KNN alone
      2. SVM alone
      3. RF alone
      4. KNN + SVM (unweighted & weighted)
      5. KNN + RF (unweighted & weighted)
      6. SVM + RF (unweighted & weighted)
      7. KNN + SVM + RF (unweighted & weighted)
    """
    combinations: list[dict[str, Any]] = [
        {"name": "KNN Alone", "models": ["KNN"], "type": "individual"},
        {"name": "SVM Alone", "models": ["SVM"], "type": "individual"},
        {"name": "RF Alone", "models": ["Random Forest"], "type": "individual"},
        {"name": "KNN + SVM (Unweighted)", "models": ["KNN", "SVM"], "type": "pairwise_unweighted"},
        {"name": "KNN + SVM (Weighted)", "models": ["KNN", "SVM"], "type": "pairwise_weighted"},
        {"name": "KNN + RF (Unweighted)", "models": ["KNN", "Random Forest"], "type": "pairwise_unweighted"},
        {"name": "KNN + RF (Weighted)", "models": ["KNN", "Random Forest"], "type": "pairwise_weighted"},
        {"name": "SVM + RF (Unweighted)", "models": ["SVM", "Random Forest"], "type": "pairwise_unweighted"},
        {"name": "SVM + RF (Weighted)", "models": ["SVM", "Random Forest"], "type": "pairwise_weighted"},
        {"name": "Hybrid Ensemble (Unweighted)", "models": ["KNN", "SVM", "Random Forest"], "type": "triplet_unweighted"},
        {"name": "Hybrid Ensemble (Weighted)", "models": ["KNN", "SVM", "Random Forest"], "type": "triplet_weighted"},
    ]

    results: list[dict[str, Any]] = []
    combination_weights: dict[str, dict[str, float]] = {}
    combined_oof_probs: dict[str, NDArray[np.float64]] = {}

    for combo in combinations:
        c_name = combo["name"]
        c_models = combo["models"]
        c_type = combo["type"]

        if len(c_models) == 1:
            w = {c_models[0]: 1.0}
            probs = oof_probs[c_models[0]]
        elif "unweighted" in c_type:
            w = {m: 1.0 / len(c_models) for m in c_models}
            probs = np.mean([oof_probs[m] for m in c_models], axis=0)
        else:  # weighted
            w = optimize_ensemble_weights(y_true, oof_probs, c_models)
            probs = np.sum([w[m] * oof_probs[m] for m in c_models], axis=0)

        combination_weights[c_name] = w
        combined_oof_probs[c_name] = probs

        m = calculate_metrics(y_true, probs)
        results.append({
            "combination": c_name,
            "models": " + ".join(c_models),
            "type": c_type,
            "weights": json.dumps({k: round(v, 4) for k, v in w.items()}),
            "roc_auc": m["roc_auc"],
            "pr_auc": m["pr_auc"],
            "f1": m["f1"],
            "recall": m["recall"],
            "specificity": m["specificity"],
            "precision": m["precision"],
            "brier": m["brier"],
        })

    ablation_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
    return ablation_df, {"weights": combination_weights, "probs": combined_oof_probs}


def compute_diversity_analysis(
    y_true: NDArray[np.int_],
    oof_probs: dict[str, NDArray[np.float64]],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Computes Pearson/Spearman probability correlations, disagreement rates,
    and double-fault metrics to mathematically prove learner complementarity.
    """
    models = ["KNN", "SVM", "Random Forest"]
    prob_df = pd.DataFrame({m: oof_probs[m] for m in models})
    pearson_corr = prob_df.corr(method="pearson")
    spearman_corr = prob_df.corr(method="spearman")

    # Binary predictions at threshold 0.5
    preds = {m: (oof_probs[m] >= 0.5).astype(int) for m in models}

    # Pairwise disagreement & double fault
    diversity_rows: list[dict[str, Any]] = []
    for i, m1 in enumerate(models):
        for j, m2 in enumerate(models):
            if i < j:
                disagreement = float(np.mean(preds[m1] != preds[m2]))
                # Double fault: both predict incorrectly
                double_fault = float(np.mean((preds[m1] != y_true) & (preds[m2] != y_true)))
                diversity_rows.append({
                    "pair": f"{m1} vs {m2}",
                    "pearson_correlation": float(cast(Any, pearson_corr.loc[m1, m2])),
                    "spearman_correlation": float(cast(Any, spearman_corr.loc[m1, m2])),
                    "disagreement_rate": disagreement,
                    "double_fault_rate": double_fault,
                })

    return pearson_corr, spearman_corr, pd.DataFrame(diversity_rows)


def run_seed_stability_study(
    X: pd.DataFrame,
    y: pd.Series,
    best_configs: dict[str, Any],
    ensemble_weights: dict[str, float],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Evaluates repeated 5-fold CV across 5 random seeds (11, 22, 33, 44, 55)
    for all key ablation candidates to compute empirical mean, std, and stability score.
    """
    candidates = [
        "KNN Alone",
        "SVM Alone",
        "RF Alone",
        "KNN + SVM",
        "KNN + RF",
        "SVM + RF",
        "Hybrid Ensemble (Unweighted)",
        "Hybrid Ensemble (Weighted)",
    ]

    rows: list[dict[str, Any]] = []
    y_arr = y.to_numpy()

    for seed in STABILITY_SEEDS:
        oof_p, _ = generate_oof_predictions(X, y, best_configs, cv_seed=seed)
        probs_map: dict[str, NDArray[np.float64]] = {
            "KNN Alone": oof_p["KNN"],
            "SVM Alone": oof_p["SVM"],
            "RF Alone": oof_p["Random Forest"],
            "KNN + SVM": 0.5 * oof_p["KNN"] + 0.5 * oof_p["SVM"],
            "KNN + RF": 0.5 * oof_p["KNN"] + 0.5 * oof_p["Random Forest"],
            "SVM + RF": 0.5 * oof_p["SVM"] + 0.5 * oof_p["Random Forest"],
            "Hybrid Ensemble (Unweighted)": (oof_p["KNN"] + oof_p["SVM"] + oof_p["Random Forest"]) / 3.0,
            "Hybrid Ensemble (Weighted)": (
                ensemble_weights["KNN"] * oof_p["KNN"] +
                ensemble_weights["SVM"] * oof_p["SVM"] +
                ensemble_weights["Random Forest"] * oof_p["Random Forest"]
            ),
        }

        for cand in candidates:
            m = calculate_metrics(y_arr, probs_map[cand])
            rows.append({
                "candidate": cand,
                "seed": seed,
                "roc_auc": m["roc_auc"],
                "pr_auc": m["pr_auc"],
                "f1": m["f1"],
                "recall": m["recall"],
                "specificity": m["specificity"],
                "brier": m["brier"],
            })

    detail_df = pd.DataFrame(rows)
    summary_df = detail_df.groupby("candidate", as_index=False).agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        f1_mean=("f1", "mean"),
        recall_mean=("recall", "mean"),
        specificity_mean=("specificity", "mean"),
        brier_mean=("brier", "mean"),
    )
    summary_df["stability_score"] = summary_df["roc_auc_mean"] - summary_df["roc_auc_std"].fillna(0)
    summary_df = summary_df.sort_values("stability_score", ascending=False)
    return detail_df, summary_df


# -----------------------------------------------------------------------------
# Visualizations
# -----------------------------------------------------------------------------
def generate_ablation_figure(ablation_df: pd.DataFrame) -> None:
    """
    Generates a 4-panel publication-grade ablation comparison figure.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    df_plot = ablation_df.copy()

    # Shorten names for cleaner display
    df_plot["short_name"] = df_plot["combination"].str.replace("Hybrid Ensemble", "Hybrid Ens")

    # Panel 1: ROC-AUC & PR-AUC
    ax1 = axes[0, 0]
    x = np.arange(len(df_plot))
    width = 0.35
    ax1.bar(x - width/2, df_plot["roc_auc"], width, label="ROC-AUC (Primary)", color="#2b5c8f")
    ax1.bar(x + width/2, df_plot["pr_auc"], width, label="PR-AUC", color="#e27c38")
    ax1.set_xticks(x)
    ax1.set_xticklabels(df_plot["short_name"], rotation=40, ha="right", fontsize=9)
    ax1.set_ylim(0.90, 1.00)
    ax1.set_title("Discrimination: ROC-AUC vs PR-AUC Across Configurations", fontsize=12, fontweight="bold")
    ax1.legend(loc="lower right")
    ax1.grid(axis="y", linestyle="--", alpha=0.5)

    # Panel 2: Clinical Metrics (Recall, Specificity, F1)
    ax2 = axes[0, 1]
    width3 = 0.25
    ax2.bar(x - width3, df_plot["recall"], width3, label="Recall / Sensitivity", color="#439775")
    ax2.bar(x, df_plot["specificity"], width3, label="Specificity", color="#6c5b7b")
    ax2.bar(x + width3, df_plot["f1"], width3, label="F1 Score", color="#d65f5f")
    ax2.set_xticks(x)
    ax2.set_xticklabels(df_plot["short_name"], rotation=40, ha="right", fontsize=9)
    ax2.set_ylim(0.85, 1.00)
    ax2.set_title("Clinical Balance: Sensitivity vs Specificity vs F1", fontsize=12, fontweight="bold")
    ax2.legend(loc="lower right")
    ax2.grid(axis="y", linestyle="--", alpha=0.5)

    # Panel 3: Probability Quality (Brier Score Loss - lower is better)
    ax3 = axes[1, 0]
    colors = ["#c0392b" if "Alone" in name else "#27ae60" for name in df_plot["combination"]]
    ax3.bar(df_plot["short_name"], df_plot["brier"], color=colors, width=0.55)
    ax3.set_xticklabels(df_plot["short_name"], rotation=40, ha="right", fontsize=9)
    ax3.set_title("Probability Quality: Brier Score Loss (Lower = Better)", fontsize=12, fontweight="bold")
    ax3.set_ylabel("Brier Score Loss")
    ax3.grid(axis="y", linestyle="--", alpha=0.5)

    # Panel 4: Metric Spider / Ranking Summary
    ax4 = axes[1, 1]
    sorted_df = df_plot.sort_values("roc_auc", ascending=True)
    ax4.barh(sorted_df["short_name"], sorted_df["roc_auc"], color="#34495e", height=0.6)
    for i, v in enumerate(sorted_df["roc_auc"]):
        ax4.text(v - 0.015, i, f"{v:.4f}", va="center", ha="right", color="white", fontweight="bold", fontsize=9)
    ax4.set_xlim(0.94, 1.00)
    ax4.set_title("Overall Leaderboard Ranked by ROC-AUC", fontsize=12, fontweight="bold")
    ax4.set_xlabel("Out-of-Fold ROC-AUC")
    ax4.grid(axis="x", linestyle="--", alpha=0.5)

    plt.suptitle("Phase 4: Ensemble Ablation Study & Systematic Component Comparison", fontsize=15, fontweight="bold", y=0.995)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_ablation_comparison.png", dpi=300, bbox_inches="tight")
    plt.close()


def generate_diversity_figure(
    y_true: NDArray[np.int_],
    oof_probs: dict[str, NDArray[np.float64]],
    pearson_corr: pd.DataFrame,
) -> None:
    """
    Generates probability correlation heatmap, scatter matrix, and error distribution.
    """
    fig = plt.figure(figsize=(16, 6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1.2, 1])

    # Subplot 1: Correlation Heatmap
    ax1 = fig.add_subplot(gs[0])
    sns.heatmap(
        pearson_corr,
        annot=True,
        fmt=".4f",
        cmap="Blues",
        vmin=0.7,
        vmax=1.0,
        square=True,
        cbar_kws={"shrink": 0.8},
        ax=ax1,
    )
    ax1.set_title("OOF Probability Pearson Correlation", fontsize=12, fontweight="bold")

    # Subplot 2: 2D Density / Scatter KNN vs RF and SVM vs RF
    ax2 = fig.add_subplot(gs[1])
    ax2.scatter(oof_probs["KNN"], oof_probs["Random Forest"], c=y_true, cmap="coolwarm", alpha=0.4, edgecolors="none", s=30, label="KNN vs RF")
    ax2.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect Agreement")
    ax2.set_xlabel("KNN Probability", fontsize=11)
    ax2.set_ylabel("Random Forest Probability", fontsize=11)
    ax2.set_title("Learner Complementarity: KNN vs RF Probabilities", fontsize=12, fontweight="bold")
    ax2.legend(loc="upper left")
    ax2.grid(True, linestyle="--", alpha=0.4)

    # Subplot 3: Disagreement & Double Fault breakdown
    ax3 = fig.add_subplot(gs[2])
    models = ["KNN", "SVM", "Random Forest"]
    preds = {m: (oof_probs[m] >= 0.5).astype(int) for m in models}

    # Error classification
    err_knn = (preds["KNN"] != y_true)
    err_svm = (preds["SVM"] != y_true)
    err_rf = (preds["Random Forest"] != y_true)

    all_wrong = int(np.sum(err_knn & err_svm & err_rf))
    two_wrong = int(np.sum((err_knn & err_svm & ~err_rf) | (err_knn & ~err_svm & err_rf) | (~err_knn & err_svm & err_rf)))
    one_wrong = int(np.sum((err_knn & ~err_svm & ~err_rf) | (~err_knn & err_svm & ~err_rf) | (~err_knn & ~err_svm & err_rf)))
    all_correct = int(np.sum(~err_knn & ~err_svm & ~err_rf))

    categories = ["All Correct\n(Consensus)", "1 Learner Wrong\n(Ensemble Recovers)", "2 Learners Wrong", "All 3 Wrong\n(Double Fault)"]
    counts = [all_correct, one_wrong, two_wrong, all_wrong]
    bar_colors = ["#27ae60", "#2980b9", "#e67e22", "#c0392b"]

    bars = ax3.bar(categories, counts, color=bar_colors, width=0.6)
    for bar in bars:
        h = bar.get_height()
        pct = (h / len(y_true)) * 100
        ax3.text(bar.get_x() + bar.get_width()/2.0, h + 5, f"{h}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax3.set_ylim(0, max(counts) * 1.15)
    ax3.set_title("Ensemble Error Complementarity Breakdown", fontsize=12, fontweight="bold")
    ax3.set_ylabel("Patient Sample Count")
    ax3.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_ensemble_diversity_correlation.png", dpi=300, bbox_inches="tight")
    plt.close()


def generate_roc_pr_figure(
    y_true: NDArray[np.int_],
    oof_probs: dict[str, NDArray[np.float64]],
    combo_probs: dict[str, NDArray[np.float64]],
) -> None:
    """
    Plots ROC Curves and Precision-Recall Curves for individual learners vs hybrid ensemble.
    """
    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))

    plot_entries = [
        ("KNN", oof_probs["KNN"], "#8e44ad", ":"),
        ("SVM", oof_probs["SVM"], "#d35400", "-."),
        ("Random Forest", oof_probs["Random Forest"], "#27ae60", "--"),
        ("Hybrid Ensemble (Unweighted)", combo_probs["Hybrid Ensemble (Unweighted)"], "#2980b9", "-"),
        ("Hybrid Ensemble (Weighted)", combo_probs["Hybrid Ensemble (Weighted)"], "#c0392b", "-"),
    ]

    for name, p, color, ls in plot_entries:
        fpr, tpr, _ = roc_curve(y_true, p)
        roc_auc = roc_auc_score(y_true, p)
        lw = 2.5 if "Hybrid" in name else 1.5
        ax_roc.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.4f})", color=color, linestyle=ls, linewidth=lw)

        precision, recall, _ = precision_recall_curve(y_true, p)
        pr_auc = average_precision_score(y_true, p)
        ax_pr.plot(recall, precision, label=f"{name} (PR-AUC = {pr_auc:.4f})", color=color, linestyle=ls, linewidth=lw)

    # Formatting ROC
    ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Chance (AUC = 0.5000)")
    ax_roc.set_xlim([0.0, 1.0])
    ax_roc.set_ylim([0.0, 1.02])
    ax_roc.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
    ax_roc.set_ylabel("True Positive Rate (Sensitivity / Recall)", fontsize=11)
    ax_roc.set_title("Receiver Operating Characteristic (ROC) Curves", fontsize=12, fontweight="bold")
    ax_roc.legend(loc="lower right", fontsize=9)
    ax_roc.grid(True, linestyle="--", alpha=0.4)

    # Formatting PR
    baseline_pr = float(np.mean(y_true))
    ax_pr.axhline(baseline_pr, color="gray", linestyle="--", alpha=0.6, label=f"Prevalence Baseline ({baseline_pr:.2f})")
    ax_pr.set_xlim([0.0, 1.0])
    ax_pr.set_ylim([0.0, 1.02])
    ax_pr.set_xlabel("Recall (Sensitivity)", fontsize=11)
    ax_pr.set_ylabel("Precision (Positive Predictive Value)", fontsize=11)
    ax_pr.set_title("Precision-Recall (PR) Curves", fontsize=12, fontweight="bold")
    ax_pr.legend(loc="lower left", fontsize=9)
    ax_pr.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_roc_pr_comparison.png", dpi=300, bbox_inches="tight")
    plt.close()


# -----------------------------------------------------------------------------
# Main Execution Pipeline
# -----------------------------------------------------------------------------
def main() -> None:
    print("=" * 80)
    print("PHASE 4: RESEARCH CENTERPIECE - HYBRID ENSEMBLE & ABLATION STUDY")
    print("Steps 17 & 18 | Strict Anti-Leakage Protocol (Test Set Locked)")
    print("=" * 80)

    # 1. Load splits
    X_train, y_train, X_val, y_val = load_training_and_val_data()
    print(f"\n[1/7] Data Loaded:")
    print(f"  - Training samples: {len(X_train)} (Positive: {y_train.sum()}, Negative: {(y_train == 0).sum()})")
    print(f"  - Validation samples: {len(X_val)} (Holdout checkpoint only)")
    print(f"  - Test partition: STRICTLY LOCKED and NOT ACCESSED")

    # 2. Load best configurations
    best_configs = load_shortlisted_configs()
    print("\n[2/7] Phase 3 Shortlisted Configurations Loaded:")
    for m, cfg in best_configs.items():
        if m in ["KNN", "SVM", "Random Forest"]:
            print(f"  * {m:14s}: Strategy = {cfg['strategy']:12s} | Params = {cfg['params']}")

    # 3. Generate Out-of-Fold (OOF) predictions
    print("\n[3/7] Generating 5-Fold Stratified OOF Probabilities on Training Data...")
    oof_probs, fold_metrics = generate_oof_predictions(X_train, y_train, best_configs)
    y_arr = y_train.to_numpy()

    # Step 17: Weight Optimization
    print("\n[4/7] Step 17: Hybrid Ensemble Weight Optimization...")
    opt_weights = optimize_ensemble_weights(y_arr, oof_probs, ["KNN", "SVM", "Random Forest"])
    print(f"  - Optimal Weights (Minimizing OOF Brier Score):")
    for m, w in opt_weights.items():
        print(f"    * {m:14s}: {w:.4f} ({w*100:.1f}%)")

    # Save weights
    with open(OUTPUT_DIR / "ensemble_weights.json", "w", encoding="utf-8") as f:
        json.dump(opt_weights, f, indent=2)

    # 4. Step 18: Systematic Ablation Study
    print("\n[5/7] Step 18: Systematic Ensemble Ablation Study Across 7 Combinations...")
    ablation_df, combo_data = run_ablation_study(y_arr, oof_probs)
    ablation_df.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)

    print("\n--- Ablation Leaderboard (Ranked by OOF ROC-AUC) ---")
    cols_display = ["combination", "roc_auc", "pr_auc", "f1", "recall", "specificity", "brier"]
    print(ablation_df[cols_display].to_string(index=False))

    # Save component vs ensemble comparison table (Step 17 expected output)
    comparison_filter = ablation_df["combination"].isin([
        "KNN Alone", "SVM Alone", "RF Alone",
        "Hybrid Ensemble (Unweighted)", "Hybrid Ensemble (Weighted)",
    ])
    comparison_df = ablation_df[comparison_filter].copy()
    comparison_df.to_csv(OUTPUT_DIR / "ensemble_comparison.csv", index=False)

    # 5. Diversity and Complementarity Analysis
    print("\n[6/7] Computing Diversity & Model Complementarity Analysis...")
    pearson_corr, spearman_corr, diversity_df = compute_diversity_analysis(y_arr, oof_probs)
    diversity_df.to_csv(OUTPUT_DIR / "prediction_correlations.csv", index=False)
    print("\n--- Pairwise Model Diversity & Error Independence ---")
    print(diversity_df.to_string(index=False))

    # Save comprehensive OOF probabilities dataframe (vital for Phase 5 calibration & thresholding)
    oof_export = pd.DataFrame({"target": y_arr})
    for m in ["KNN", "SVM", "Random Forest"]:
        oof_export[f"prob_{m.lower().replace(' ', '_')}"] = oof_probs[m]
    oof_export["prob_hybrid_unweighted"] = combo_data["probs"]["Hybrid Ensemble (Unweighted)"]
    oof_export["prob_hybrid_weighted"] = combo_data["probs"]["Hybrid Ensemble (Weighted)"]
    oof_export.to_csv(OUTPUT_DIR / "oof_probabilities.csv", index=False)
    print(f"\nSaved OOF probabilities to {OUTPUT_DIR / 'oof_probabilities.csv'}")

    # Multi-seed stability across 5 seeds
    print("\n[7/7] Multi-Seed Stability Verification (Seeds: 11, 22, 33, 44, 55)...")
    stability_detail, stability_summary = run_seed_stability_study(
        X_train, y_train, best_configs, opt_weights
    )
    stability_detail.to_csv(OUTPUT_DIR / "ablation_stability_by_seed.csv", index=False)
    stability_summary.to_csv(OUTPUT_DIR / "ablation_stability_summary.csv", index=False)

    print("\n--- Stability Summary Across 5 Repeated Seeds ---")
    print(stability_summary[["candidate", "roc_auc_mean", "roc_auc_std", "stability_score"]].to_string(index=False))

    # Visualizations
    print("\nGenerating Publication Figures in models/phase4/...")
    generate_ablation_figure(ablation_df)
    generate_diversity_figure(y_arr, oof_probs, pearson_corr)
    generate_roc_pr_figure(y_arr, oof_probs, combo_data["probs"])
    print("  [OK] fig_ablation_comparison.png")
    print("  [OK] fig_ensemble_diversity_correlation.png")
    print("  [OK] fig_roc_pr_comparison.png")

    # Fit authoritative production ensemble model on full training data
    print("\nFitting Authoritative Hybrid Ensemble on Full Training Data...")
    ensemble_clf = HybridEnsembleClassifier(weights=opt_weights, best_configs=best_configs)
    ensemble_clf.fit(X_train, y_train)

    # Evaluate on holdout validation set (non-tuning checkpoint)
    val_probs = ensemble_clf.predict_proba(X_val)[:, 1]
    val_metrics = calculate_metrics(y_val, val_probs)
    print("\nHoldout Validation Checkpoint Metrics (val.csv):")
    for k, v in val_metrics.items():
        print(f"  * {k:12s}: {v:.4f}")

    # Serialize fitted ensemble model
    model_artifact_path = OUTPUT_DIR / "hybrid_ensemble_model.pkl"
    joblib.dump(ensemble_clf, model_artifact_path)
    print(f"\nAuthoritative Hybrid Ensemble serialized to: {model_artifact_path}")

    # Write Phase 4 Decision & Manifest
    decision_payload = {
        "phase": 4,
        "steps": [17, 18],
        "experiment_ids": ["E05", "E06"],
        "primary_metric": "ROC-AUC",
        "weights": opt_weights,
        "best_ensemble_configuration": "Hybrid Ensemble (Weighted)",
        "oof_roc_auc_weighted": float(ablation_df.loc[ablation_df['combination'] == 'Hybrid Ensemble (Weighted)', 'roc_auc'].iloc[0]),
        "oof_roc_auc_unweighted": float(ablation_df.loc[ablation_df['combination'] == 'Hybrid Ensemble (Unweighted)', 'roc_auc'].iloc[0]),
        "oof_roc_auc_best_single (RF)": float(ablation_df.loc[ablation_df['combination'] == 'RF Alone', 'roc_auc'].iloc[0]),
        "stability_score_weighted": float(stability_summary.loc[stability_summary['candidate'] == 'Hybrid Ensemble (Weighted)', 'stability_score'].iloc[0]),
        "test_split_integrity": "Untouched / Locked (zero access)",
        "decision": "Proceed to Phase 5 (Steps 19-21: Calibration, Threshold Analysis, Locked Test Eval) using the validated Hybrid Ensemble.",
    }
    with open(OUTPUT_DIR / "phase4_decision.json", "w", encoding="utf-8") as f:
        json.dump(decision_payload, f, indent=2)

    print("\nPhase 4 (Steps 17–18) completed successfully!")
    print(f"All outputs and figures stored in: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()



Writing phase4_ensemble_ablation.py


In [6]:
%%writefile phase5_calibration_final_eval.py
# Phase 5 (Steps 19–21): Calibration, threshold analysis, final locked test evaluation
# Strict anti-leakage protocol:
#   * Steps 19–20 operate exclusively on splits/train.csv (Out-Of-Fold) and splits/val.csv.
#   * Calibration methods are compared via internal cross-validation; final calibrators are
#     fitted on validation data only (never on the test set).
#   * The decision threshold is selected exclusively from validation evidence.
#   * splits/test.csv is read EXACTLY ONCE, in Step 21, after the pipeline is locked.
# Outputs are written to models/phase5/.

from __future__ import annotations

import hashlib
import json
import sys
import time
import warnings
from pathlib import Path
from typing import Any, Literal, cast

# Ensure stdout uses UTF-8 or safe encoding
if hasattr(sys.stdout, "reconfigure"):
    try:
        getattr(sys.stdout, "reconfigure")(encoding="utf-8")
    except Exception:
        pass

sys.path.insert(0, str(Path(__file__).resolve().parent))
import phase4_ensemble_ablation as p4  # noqa: E402  (reuses Phase 4 building blocks)

import joblib  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import seaborn as sns  # noqa: E402
from numpy.typing import ArrayLike, NDArray  # noqa: E402
from sklearn.isotonic import IsotonicRegression  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold  # noqa: E402

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# Configuration and Reproducibility Setup
# -----------------------------------------------------------------------------
SEED = 42
CV_FOLDS = 5
N_BINS = 10
BOOTSTRAP_ITERATIONS = 2000
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
CANDIDATES = ["KNN", "SVM", "Random Forest", "Hybrid Ensemble (Weighted)"]
ENSEMBLE_NAME = "Hybrid Ensemble (Weighted)"
CALIBRATION_METHODS = ("none", "sigmoid", "isotonic")
CalibrationMethod = Literal["none", "sigmoid", "isotonic"]

ROOT = Path(__file__).resolve().parent
PHASE4_DIR = ROOT / "models" / "phase4"
OUTPUT_DIR = ROOT / "models" / "phase5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Module-level guard proving the locked test split is consumed exactly once.
_TEST_SPLIT_ACCESSED = False


# -----------------------------------------------------------------------------
# Calibrators (public scikit-learn primitives only; fully serializable)
# -----------------------------------------------------------------------------
class IdentityCalibrator:
    """No-op calibrator (raw model probabilities)."""

    name = "none"

    def fit(self, probs: ArrayLike, y_true: ArrayLike) -> "IdentityCalibrator":
        return self

    def transform(self, probs: ArrayLike) -> NDArray[np.float64]:
        return np.asarray(probs, dtype=float)

    def params(self) -> dict[str, Any]:
        return {"method": "none"}


class PlattCalibrator:
    """Platt scaling (sigmoid): logistic regression on the logit of raw scores."""

    name = "sigmoid"

    def __init__(self) -> None:
        self._lr: LogisticRegression | None = None

    def fit(self, probs: ArrayLike, y_true: ArrayLike) -> "PlattCalibrator":
        z = self._logit(probs)
        self._lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
        self._lr.fit(z, np.asarray(y_true, dtype=int))
        return self

    def transform(self, probs: ArrayLike) -> NDArray[np.float64]:
        if self._lr is None:
            raise RuntimeError("PlattCalibrator must be fitted before transform().")
        z = self._logit(probs)
        return self._lr.predict_proba(z)[:, 1]

    @staticmethod
    def _logit(probs: ArrayLike) -> NDArray[np.float64]:
        p = np.clip(np.asarray(probs, dtype=float), 1e-6, 1.0 - 1e-6)
        return np.log(p / (1.0 - p)).reshape(-1, 1)

    def params(self) -> dict[str, Any]:
        if self._lr is None:
            raise RuntimeError("PlattCalibrator must be fitted before params().")
        return {
            "method": "sigmoid",
            "coef": float(self._lr.coef_[0, 0]),
            "intercept": float(self._lr.intercept_[0]),
        }


class IsotonicCalibrator:
    """Isotonic regression calibration (non-parametric monotonic mapping)."""

    name = "isotonic"

    def __init__(self) -> None:
        self._iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)

    def fit(self, probs: ArrayLike, y_true: ArrayLike) -> "IsotonicCalibrator":
        self._iso.fit(np.asarray(probs, dtype=float), np.asarray(y_true, dtype=float))
        return self

    def transform(self, probs: ArrayLike) -> NDArray[np.float64]:
        return np.asarray(self._iso.predict(np.asarray(probs, dtype=float)), dtype=float)

    def params(self) -> dict[str, Any]:
        f = cast(Any, self._iso).f_
        return {
            "method": "isotonic",
            "thresholds": [float(v) for v in np.asarray(f.x)],
            "values": [float(v) for v in np.asarray(f.y)],
        }


def build_calibrator(method: CalibrationMethod) -> IdentityCalibrator | PlattCalibrator | IsotonicCalibrator:
    """Factory returning the requested calibrator instance."""
    if method == "none":
        return IdentityCalibrator()
    if method == "sigmoid":
        return PlattCalibrator()
    if method == "isotonic":
        return IsotonicCalibrator()
    raise ValueError(f"Unknown calibration method: {method}")


# -----------------------------------------------------------------------------
# Calibration & Metric Helpers
# -----------------------------------------------------------------------------
def expected_calibration_error(
    y_true: ArrayLike,
    probs: ArrayLike,
    n_bins: int = N_BINS,
) -> float:
    """Equal-width binned Expected Calibration Error (ECE)."""
    labels = np.asarray(y_true, dtype=int)
    scores = np.asarray(probs, dtype=float)
    bin_idx = np.clip((scores * n_bins).astype(int), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        mask = bin_idx == b
        count = int(np.count_nonzero(mask))
        if count == 0:
            continue
        ece += (count / len(labels)) * abs(float(labels[mask].mean()) - float(scores[mask].mean()))
    return float(ece)


def calibration_metrics(y_true: ArrayLike, probs: ArrayLike) -> dict[str, float]:
    """Brier score, ECE, and log loss for a probability vector."""
    labels = np.asarray(y_true, dtype=int)
    scores = np.clip(np.asarray(probs, dtype=float), 1e-6, 1.0 - 1e-6)
    return {
        "brier": float(brier_score_loss(labels, scores)),
        "ece": expected_calibration_error(labels, scores),
        "log_loss": float(log_loss(labels, scores, labels=[0, 1])),
    }


def reliability_curve(
    y_true: ArrayLike,
    probs: ArrayLike,
    n_bins: int = N_BINS,
) -> pd.DataFrame:
    """Binned reliability curve data (mean predicted probability vs observed frequency)."""
    labels = np.asarray(y_true, dtype=int)
    scores = np.asarray(probs, dtype=float)
    bin_idx = np.clip((scores * n_bins).astype(int), 0, n_bins - 1)
    rows: list[dict[str, float]] = []
    for b in range(n_bins):
        mask = bin_idx == b
        if int(np.count_nonzero(mask)) == 0:
            continue
        rows.append({
            "bin_low": b / n_bins,
            "bin_high": (b + 1) / n_bins,
            "mean_predicted": float(scores[mask].mean()),
            "observed_frequency": float(labels[mask].mean()),
            "count": int(np.count_nonzero(mask)),
        })
    return pd.DataFrame(rows)


def classification_metrics_at_threshold(
    y_true: ArrayLike,
    probs: ArrayLike,
    threshold: float,
) -> dict[str, float]:
    """Full confusion-matrix-based metric set at an explicit decision threshold."""
    labels = np.asarray(y_true, dtype=int)
    scores = np.asarray(probs, dtype=float)
    predictions = (scores >= threshold).astype(int)

    tp = int(np.count_nonzero((predictions == 1) & (labels == 1)))
    fp = int(np.count_nonzero((predictions == 1) & (labels == 0)))
    tn = int(np.count_nonzero((predictions == 0) & (labels == 0)))
    fn = int(np.count_nonzero((predictions == 0) & (labels == 1)))
    total = tp + fp + tn + fn

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    return {
        "threshold": float(threshold),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision": precision,
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "balanced_accuracy": (recall + specificity) / 2.0,
        "youden_j": recall + specificity - 1.0,
        "accuracy": (tp + tn) / total if total > 0 else 0.0,
        "predicted_positive_rate": (tp + fp) / total if total > 0 else 0.0,
    }


def bootstrap_ci(
    y_true: NDArray[np.int_],
    probs: NDArray[np.float64],
    metric_fn: Any,
    n_iterations: int = BOOTSTRAP_ITERATIONS,
    seed: int = SEED,
) -> tuple[float, float]:
    """Percentile bootstrap 95% confidence interval for an arbitrary probability metric."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    values: list[float] = []
    for _ in range(n_iterations):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        values.append(float(metric_fn(y_true[idx], probs[idx])))
    lower, upper = np.percentile(values, [2.5, 97.5])
    return float(lower), float(upper)


# -----------------------------------------------------------------------------
# Data Loading & Probability Generation (train/val only until Step 21)
# -----------------------------------------------------------------------------
def load_training_and_val_data() -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """Loads training and validation splits. Test split is NOT touched here."""
    train = pd.read_csv(ROOT / "splits" / "train.csv")
    val = pd.read_csv(ROOT / "splits" / "val.csv")
    if "target" not in train.columns or "target" not in val.columns:
        raise ValueError("Both train.csv and val.csv must contain a 'target' column")
    X_train = train.drop(columns=["target"])
    y_train = train["target"].astype(int)
    X_val = val.drop(columns=["target"])
    y_val = val["target"].astype(int)
    return X_train, y_train, X_val, y_val


def build_probability_sets(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    best_configs: dict[str, Any],
    weights: dict[str, float],
) -> tuple[dict[str, NDArray[np.float64]], dict[str, NDArray[np.float64]], Any, bool]:
    """
    Produces honest probability sets for every calibration candidate:
      * oof_probs: Out-Of-Fold probabilities on the training data (Phase 4 protocol,
        seed 42, identical to the probabilities saved by Phase 4).
      * val_probs: probabilities on the validation set from candidates fitted on the
        FULL training data (the Phase 4 authoritative checkpoint protocol).
    Returns (oof_probs, val_probs, fitted_ensemble, oof_reproduction_ok).
    """
    # --- Out-of-fold probabilities on training data (Phase 4 protocol, seed 42) ---
    oof_components, _ = p4.generate_oof_predictions(X_train, y_train, best_configs, cv_seed=SEED)
    oof_probs: dict[str, NDArray[np.float64]] = {
        "KNN": oof_components["KNN"],
        "SVM": oof_components["SVM"],
        "Random Forest": oof_components["Random Forest"],
        ENSEMBLE_NAME: (
            weights["KNN"] * oof_components["KNN"]
            + weights["SVM"] * oof_components["SVM"]
            + weights["Random Forest"] * oof_components["Random Forest"]
        ),
    }

    # --- Integrity check against Phase 4 saved OOF probabilities ---
    oof_reproduction_ok = True
    phase4_oof_path = PHASE4_DIR / "oof_probabilities.csv"
    if phase4_oof_path.exists():
        saved = pd.read_csv(phase4_oof_path)
        col_map = {"KNN": "prob_knn", "SVM": "prob_svm", "Random Forest": "prob_random_forest"}
        for cand, col in col_map.items():
            if col in saved.columns:
                diff = float(np.max(np.abs(saved[col].to_numpy() - oof_probs[cand])))
                oof_reproduction_ok &= diff < 1e-6
                print(f"  [integrity] OOF reproduction {cand}: max|diff| = {diff:.2e}")

    # --- Validation probabilities from candidates fitted on the full training set ---
    val_probs: dict[str, NDArray[np.float64]] = {}
    for cand in ["KNN", "SVM", "Random Forest"]:
        pipeline = p4.build_pipeline(X_train, cast(p4.ModelName, cand), best_configs)
        pipeline.fit(X_train, y_train)
        val_probs[cand] = pipeline.predict_proba(X_val)[:, 1]

    # Authoritative hybrid ensemble: refit deterministically (RF/SVC use fixed seeds)
    # with the exact Phase 4 configuration and optimized weights.
    ensemble = p4.HybridEnsembleClassifier(weights=weights, best_configs=best_configs)
    ensemble.fit(X_train, y_train)
    val_probs[ENSEMBLE_NAME] = ensemble.predict_proba(X_val)[:, 1]
    return oof_probs, val_probs, ensemble, bool(oof_reproduction_ok)


# -----------------------------------------------------------------------------
# Step 19: Probability Calibration
# -----------------------------------------------------------------------------
def cross_validated_calibration_comparison(
    probs: NDArray[np.float64],
    y_true: NDArray[np.int_],
    cv_seed: int = SEED,
) -> pd.DataFrame:
    """
    Honest, internal-CV comparison of calibration methods for one candidate.
    The calibrator is fitted on 4/5 of the (prob, y) pairs and scored on the held-out
    fifth; scores are averaged across stratified folds. No other data is touched.
    """
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=cv_seed)
    rows: list[dict[str, Any]] = []
    for method in CALIBRATION_METHODS:
        fold_brier: list[float] = []
        fold_ece: list[float] = []
        fold_logloss: list[float] = []
        for train_idx, test_idx in cv.split(np.zeros(len(y_true)), y_true):
            calibrator = build_calibrator(cast(CalibrationMethod, method))
            calibrator.fit(probs[train_idx], y_true[train_idx])
            calibrated = calibrator.transform(probs[test_idx])
            m = calibration_metrics(y_true[test_idx], calibrated)
            fold_brier.append(m["brier"])
            fold_ece.append(m["ece"])
            fold_logloss.append(m["log_loss"])
        rows.append({
            "method": method,
            "brier_mean": float(np.mean(fold_brier)),
            "brier_std": float(np.std(fold_brier)),
            "ece_mean": float(np.mean(fold_ece)),
            "log_loss_mean": float(np.mean(fold_logloss)),
        })
    return pd.DataFrame(rows).sort_values("brier_mean").reset_index(drop=True)


def run_step19_calibration_part1(
    oof_probs: dict[str, NDArray[np.float64]],
    val_probs: dict[str, NDArray[np.float64]],
    y_train_arr: NDArray[np.int_],
    y_val_arr: NDArray[np.int_],
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, str]]:
    """
    Step 19 (part 1): assess uncalibrated calibration of best individual models and
    the ensemble, then compare calibration methods via internal stratified CV on both
    OOF-train and validation probability sets.
    """
    # --- 1. Uncalibrated calibration assessment ---
    assessment_rows: list[dict[str, Any]] = []
    for cand in CANDIDATES:
        for split, y_ref, probs in [
            ("train_oof", y_train_arr, oof_probs[cand]),
            ("validation", y_val_arr, val_probs[cand]),
        ]:
            m = calibration_metrics(y_ref, probs)
            assessment_rows.append({"candidate": cand, "split": split, "calibration": "none", **m})
    assessment_df = pd.DataFrame(assessment_rows)
    assessment_df.to_csv(OUTPUT_DIR / "calibration_assessment_uncalibrated.csv", index=False)
    print("\n--- Uncalibrated Calibration Assessment (Brier / ECE / LogLoss) ---")
    print(assessment_df.to_string(index=False))

    # --- 2. Cross-validated comparison of calibration methods ---
    comparison_rows: list[pd.DataFrame] = []
    for cand in CANDIDATES:
        for split, y_ref, probs in [
            ("train_oof", y_train_arr, oof_probs[cand]),
            ("validation", y_val_arr, val_probs[cand]),
        ]:
            cmp_df = cross_validated_calibration_comparison(probs, y_ref)
            cmp_df.insert(0, "candidate", cand)
            cmp_df.insert(1, "split", split)
            comparison_rows.append(cmp_df)
    comparison_df = pd.concat(comparison_rows, ignore_index=True)
    comparison_df.to_csv(OUTPUT_DIR / "calibration_method_cv_comparison.csv", index=False)
    print("\n--- Cross-Validated Calibration Method Comparison (5-fold, per candidate) ---")
    print(comparison_df[comparison_df["split"] == "validation"].to_string(index=False))
    return assessment_df, comparison_df, {}


def run_step19_calibration_part2(
    val_probs: dict[str, NDArray[np.float64]],
    y_val_arr: NDArray[np.int_],
    comparison_df: pd.DataFrame,
) -> dict[str, Any]:
    """
    Step 19 (part 2): select the final calibration method per candidate using
    validation evidence, refit final calibrators on the FULL validation set
    (roadmap protocol), and export reliability-curve data.
    """
    chosen: dict[str, str] = {}
    val_only = comparison_df[comparison_df["split"] == "validation"]
    for cand in CANDIDATES:
        cand_rows = val_only[val_only["candidate"] == cand]
        best_row = cand_rows.sort_values("brier_mean").iloc[0]
        chosen[cand] = str(best_row["method"])

    with open(OUTPUT_DIR / "chosen_calibration.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "selection_metric": "cross-validated Brier score (5-fold stratified, within-set)",
                "selection_data": "validation probabilities (calibrators refit on the full validation set)",
                "chosen_method_per_candidate": chosen,
                "final_procedure": {
                    "candidate": ENSEMBLE_NAME,
                    "calibrator": chosen[ENSEMBLE_NAME],
                    "fitted_on": "splits/val.csv (probabilities from the Phase 4 ensemble trained on splits/train.csv)",
                },
            },
            f,
            indent=2,
        )
    print("\n--- Chosen Calibration Method Per Candidate (by CV Brier on validation) ---")
    for cand in CANDIDATES:
        print(f"  * {cand:28s}: {chosen[cand]}")

    # --- Fit final calibrators on the FULL validation set; report calibrated metrics ---
    final_calibrators: dict[str, Any] = {}
    final_rows: list[dict[str, Any]] = []
    for cand in CANDIDATES:
        method = cast(CalibrationMethod, chosen[cand])
        calibrator = build_calibrator(method)
        calibrator.fit(val_probs[cand], y_val_arr)
        final_calibrators[cand] = calibrator
        calibrated = calibrator.transform(val_probs[cand])
        m = calibration_metrics(y_val_arr, calibrated)
        final_rows.append({"candidate": cand, "calibration": method, "fitted_on": "validation", **m})
    final_df = pd.DataFrame(final_rows)
    final_df.to_csv(OUTPUT_DIR / "calibration_final_comparison.csv", index=False)
    print("\n--- Final Calibrated Validation Metrics (calibrators fitted on validation) ---")
    print(final_df.to_string(index=False))

    # --- Reliability curve data (uncalibrated vs final calibrated, validation) ---
    reliability_frames: list[pd.DataFrame] = []
    for cand in CANDIDATES:
        calibrated_probs = final_calibrators[cand].transform(val_probs[cand])
        for label, probs in [("uncalibrated", val_probs[cand]), ("calibrated", calibrated_probs)]:
            rc = reliability_curve(y_val_arr, probs)
            rc.insert(0, "curve", label)
            rc.insert(0, "candidate", cand)
            reliability_frames.append(rc)
    reliability_df = pd.concat(reliability_frames, ignore_index=True)
    reliability_df.to_csv(OUTPUT_DIR / "calibration_reliability_curves.csv", index=False)

    return {"chosen": chosen, "final_calibrators": final_calibrators, "final_df": final_df}


# -----------------------------------------------------------------------------
# Step 20: Decision-Threshold Analysis (validation evidence only)
# -----------------------------------------------------------------------------
def run_step20_threshold_analysis(
    val_probs: NDArray[np.float64],
    y_val_arr: NDArray[np.int_],
) -> dict[str, Any]:
    """
    Step 20 — Decision-threshold analysis on the CALIBRATED validation probabilities.
    Sweeps an explicit threshold grid, builds the threshold-performance table and
    trade-off visualization, and locks a documented threshold rule.
    CRITICAL: the test set is NOT used anywhere in this step.
    """
    sweep_rows: list[dict[str, float]] = []
    for t in THRESHOLD_GRID:
        row = classification_metrics_at_threshold(y_val_arr, val_probs, float(t))
        sweep_rows.append(row)
    sweep_df = pd.DataFrame(sweep_rows)
    sweep_df.to_csv(OUTPUT_DIR / "threshold_performance.csv", index=False)

    # --- Threshold rule: maximize Youden's J; ties (within 1e-3) resolved toward
    # higher sensitivity, reflecting the screening context where false negatives
    # (missed disease) are costlier than false positives (extra follow-up tests). ---
    max_j = float(sweep_df["youden_j"].max())
    tie_mask = sweep_df["youden_j"] >= max_j - 1e-3
    tied = sweep_df[tie_mask].sort_values("recall_sensitivity", ascending=False)
    best_row = tied.iloc[0]
    final_threshold = float(best_row["threshold"])

    baseline_row = sweep_df[np.isclose(sweep_df["threshold"], 0.5)].iloc[0]

    print("\n--- Threshold Sweep Highlights (calibrated validation probabilities) ---")
    display_cols = [
        "threshold", "recall_sensitivity", "specificity", "precision",
        "f1", "balanced_accuracy", "youden_j", "predicted_positive_rate",
    ]
    show_idx = list(np.linspace(0, len(sweep_df) - 1, 10).astype(int))
    print(sweep_df.iloc[show_idx][display_cols].to_string(index=False))
    print(f"\n  Default threshold 0.50 -> recall={baseline_row['recall_sensitivity']:.3f}, "
          f"specificity={baseline_row['specificity']:.3f}, F1={baseline_row['f1']:.3f}")
    print(f"  LOCKED threshold {final_threshold:.2f} -> recall={best_row['recall_sensitivity']:.3f}, "
          f"specificity={best_row['specificity']:.3f}, F1={best_row['f1']:.3f}, "
          f"balanced_acc={best_row['balanced_accuracy']:.3f}, J={best_row['youden_j']:.3f}")

    threshold_decision = {
        "locked_threshold": final_threshold,
        "selection_rule": "Maximize Youden's J (sensitivity + specificity - 1); ties within 1e-3 resolved toward higher sensitivity.",
        "operational_rationale": (
            "Heart-disease screening context: a missed case (false negative) is costlier than an "
            "unnecessary follow-up test (false positive). Youden's J balances sensitivity and "
            "specificity without assuming 0.50 is optimal; near-ties favor higher sensitivity."
        ),
        "evidence_source": "calibrated validation probabilities only (splits/val.csv); test set NOT accessed",
        "validation_operating_point": {
            k: (float(v) if isinstance(v, (int, float, np.floating, np.integer)) else v)
            for k, v in best_row.items()
        },
        "default_threshold_050_comparison": {
            k: (float(v) if isinstance(v, (int, float, np.floating, np.integer)) else v)
            for k, v in baseline_row.items()
        },
    }
    with open(OUTPUT_DIR / "threshold_decision.json", "w", encoding="utf-8") as f:
        json.dump(threshold_decision, f, indent=2)

    return {"sweep_df": sweep_df, "best_row": best_row, "final_threshold": final_threshold}


# -----------------------------------------------------------------------------
# Step 21: Locked Production Artifact & Final Untouched Test Evaluation
# -----------------------------------------------------------------------------
class CalibratedHybridEnsemble:
    """
    Production inference artifact for the locked final system (roadmap Step 21 gate:
    'the calibration procedure itself must be part of the final serialized pipeline').

    Bundles, in exact operating order:
      1. The Phase 4 hybrid soft-voting ensemble (KNN + SVM + RF, fitted on train).
      2. The chosen probability calibrator (fitted on validation data).
      3. The locked decision threshold (selected on validation evidence).
    """

    def __init__(self, ensemble: Any, calibrator: Any, threshold: float):
        self.ensemble = ensemble
        self.calibrator = calibrator
        self.threshold = float(threshold)
        self.classes_ = np.array([0, 1])

    def predict_proba(self, X: pd.DataFrame) -> NDArray[np.float64]:
        raw = self.ensemble.predict_proba(X)[:, 1]
        calibrated = np.asarray(self.calibrator.transform(raw), dtype=float)
        return np.column_stack([1.0 - calibrated, calibrated])

    def predict(self, X: pd.DataFrame) -> NDArray[np.int_]:
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)


def sha256_of_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_of_payload(payload: Any) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()


def load_production_artifact(path: Path) -> "CalibratedHybridEnsemble":
    """
    Loads the serialized production artifact from any process/module.
    Registers Phase 5 classes on __main__ if needed, because pickle references
    classes by the module in which the producing script was executed.
    """
    import __main__

    for cls in (CalibratedHybridEnsemble, IdentityCalibrator, PlattCalibrator, IsotonicCalibrator):
        if not hasattr(__main__, cls.__name__):
            setattr(__main__, cls.__name__, cls)
    return cast(CalibratedHybridEnsemble, joblib.load(path))


def run_step21_locked_test_evaluation(
    ensemble: Any,
    ensemble_calibrator: Any,
    final_threshold: float,
    calibration_method: str,
    X_val: pd.DataFrame,
    y_val_arr: NDArray[np.int_],
    val_probs_calibrated: NDArray[np.float64],
) -> dict[str, Any]:
    """
    Step 21 — Final untouched test evaluation.
    1. Freezes the locked pipeline (writes the production artifact + lock manifest).
    2. Reads splits/test.csv EXACTLY ONCE (the single authorized access in Phase 5).
    3. Applies the locked pipeline unchanged and reports the final generalization result.
    """
    global _TEST_SPLIT_ACCESSED
    if _TEST_SPLIT_ACCESSED:
        raise RuntimeError("Test split may only be accessed once per evaluation cycle.")

    # --- 1. Freeze the locked pipeline ---
    artifact_path = OUTPUT_DIR / "hybrid_ensemble_calibrated_production.pkl"
    locked_artifact = CalibratedHybridEnsemble(ensemble, ensemble_calibrator, final_threshold)
    joblib.dump(locked_artifact, artifact_path)

    calibrator_params = ensemble_calibrator.params()
    lock_manifest = {
        "locked_pipeline": {
            "model": "Hybrid Ensemble (Weighted) — Phase 4 (fitted on splits/train.csv)",
            "model_source": str(PHASE4_DIR / "hybrid_ensemble_model.pkl"),
            "calibration": calibrator_params,
            "calibrator_fitted_on": "splits/val.csv",
            "decision_threshold": final_threshold,
            "threshold_rule": "Youden's J on calibrated validation probabilities",
        },
        "input_hashes": {
            "phase4_ensemble_model.pkl": sha256_of_file(PHASE4_DIR / "hybrid_ensemble_model.pkl"),
            "ensemble_weights.json": sha256_of_file(PHASE4_DIR / "ensemble_weights.json"),
            "best_configurations.json": sha256_of_file(ROOT / "models" / "phase3" / "best_configurations.json"),
        },
        "artifact_hash_sha256": None,  # filled after artifact is written
        "artifact_path": str(artifact_path),
    }
    lock_manifest["artifact_hash_sha256"] = sha256_of_file(artifact_path)
    with open(OUTPUT_DIR / "lock_manifest.json", "w", encoding="utf-8") as f:
        json.dump(lock_manifest, f, indent=2)
    print(f"\n[lock] Production artifact written : {artifact_path}")
    print(f"[lock] Artifact SHA-256            : {lock_manifest['artifact_hash_sha256']}")

    # Artifact round-trip regression check (validates serialization integrity)
    reloaded = load_production_artifact(artifact_path)
    reload_diff = float(np.max(np.abs(reloaded.predict_proba(X_val)[:, 1] - val_probs_calibrated)))
    assert reload_diff < 1e-9, "Production artifact round-trip mismatch!"
    print(f"[lock] Artifact round-trip check   : OK (max|diff| = {reload_diff:.2e})")

    # --- 2. THE single authorized read of the untouched test split ---
    _TEST_SPLIT_ACCESSED = True
    test = pd.read_csv(ROOT / "splits" / "test.csv")
    X_test = test.drop(columns=["target"])
    y_test_arr = test["target"].to_numpy(dtype=np.int_)
    print(f"\n[test] splits/test.csv accessed EXACTLY ONCE: {len(X_test)} samples "
          f"(positive rate {y_test_arr.mean():.3f})")

    # --- 3. Apply the locked pipeline UNCHANGED ---
    raw_test_probs = ensemble.predict_proba(X_test)[:, 1]
    calibrated_test_probs = np.asarray(ensemble_calibrator.transform(raw_test_probs), dtype=float)
    test_predictions = (calibrated_test_probs >= final_threshold).astype(int)
    return _step21_collect_results(
        y_test_arr, calibrated_test_probs, raw_test_probs, test_predictions,
        ensemble, ensemble_calibrator, final_threshold, calibration_method,
        artifact_path, X_val, val_probs_calibrated,
    )


def _step21_collect_results(
    y_test_arr: NDArray[np.int_],
    calibrated_test_probs: NDArray[np.float64],
    raw_test_probs: NDArray[np.float64],
    test_predictions: NDArray[np.int_],
    ensemble: Any,
    ensemble_calibrator: Any,
    final_threshold: float,
    calibration_method: str,
    artifact_path: Path,
    X_val: pd.DataFrame,
    val_probs_calibrated: NDArray[np.float64],
) -> dict[str, Any]:
    """Computes final test metrics, exports curve data, and locks the decision record."""
    # Discrimination metrics with bootstrap 95% CIs
    roc_auc = float(roc_auc_score(y_test_arr, calibrated_test_probs))
    pr_auc = float(average_precision_score(y_test_arr, calibrated_test_probs))
    roc_lo, roc_hi = bootstrap_ci(y_test_arr, calibrated_test_probs, roc_auc_score)
    pr_lo, pr_hi = bootstrap_ci(y_test_arr, calibrated_test_probs, average_precision_score)

    # Threshold-based operating metrics and confusion matrix
    op = classification_metrics_at_threshold(y_test_arr, calibrated_test_probs, final_threshold)
    cm = confusion_matrix(y_test_arr, test_predictions, labels=[0, 1])

    # Probability-quality metrics
    q = calibration_metrics(y_test_arr, calibrated_test_probs)
    raw_q = calibration_metrics(y_test_arr, raw_test_probs)

    final_metrics: dict[str, float] = {
        "roc_auc": roc_auc,
        "roc_auc_ci95_low": roc_lo,
        "roc_auc_ci95_high": roc_hi,
        "pr_auc": pr_auc,
        "pr_auc_ci95_low": pr_lo,
        "pr_auc_ci95_high": pr_hi,
        "accuracy": op["accuracy"],
        "recall_sensitivity": op["recall_sensitivity"],
        "specificity": op["specificity"],
        "precision": op["precision"],
        "f1": op["f1"],
        "balanced_accuracy": op["balanced_accuracy"],
        "brier_calibrated": q["brier"],
        "ece_calibrated": q["ece"],
        "log_loss_calibrated": q["log_loss"],
        "brier_uncalibrated_reference": raw_q["brier"],
        "ece_uncalibrated_reference": raw_q["ece"],
    }
    metrics_df = pd.DataFrame(
        [{"metric": k, "value": float(v)} for k, v in final_metrics.items()]
    )
    metrics_df.to_csv(OUTPUT_DIR / "final_test_metrics.csv", index=False)

    # Export full curve data
    fpr, tpr, _ = roc_curve(y_test_arr, calibrated_test_probs)
    pd.DataFrame({"fpr": fpr, "tpr": tpr}).to_csv(OUTPUT_DIR / "test_roc_curve.csv", index=False)
    prec, rec, _ = precision_recall_curve(y_test_arr, calibrated_test_probs)
    pd.DataFrame({"recall": rec, "precision": prec}).to_csv(OUTPUT_DIR / "test_pr_curve.csv", index=False)
    reliability_curve(y_test_arr, calibrated_test_probs).to_csv(
        OUTPUT_DIR / "test_calibration_curve.csv", index=False
    )
    pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"]).to_csv(
        OUTPUT_DIR / "confusion_matrix.csv"
    )

    return {
        "y_test": y_test_arr,
        "probs": calibrated_test_probs,
        "raw_probs": raw_test_probs,
        "predictions": test_predictions,
        "metrics": final_metrics,
        "confusion_matrix": cm,
        "artifact_path": artifact_path,
        "threshold": final_threshold,
        "calibration_method": calibration_method,
        "ensemble": ensemble,
        "ensemble_calibrator": ensemble_calibrator,
        "X_val": X_val,
        "val_probs_calibrated": val_probs_calibrated,
    }


# -----------------------------------------------------------------------------
# Visualizations
# -----------------------------------------------------------------------------
def generate_calibration_figures(
    val_probs: dict[str, NDArray[np.float64]],
    y_val_arr: NDArray[np.int_],
    final_calibrators: dict[str, Any],
) -> None:
    """Reliability diagrams (uncalibrated vs calibrated) + Brier score comparison."""
    # Panel figure: one reliability diagram per candidate
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    for ax, cand in zip(axes.ravel(), CANDIDATES):
        calibrated = final_calibrators[cand].transform(val_probs[cand])
        rc_raw = reliability_curve(y_val_arr, val_probs[cand])
        rc_cal = reliability_curve(y_val_arr, calibrated)
        ax.plot([0, 1], [0, 1], "k--", alpha=0.6, linewidth=1, label="Perfect calibration")
        ax.plot(rc_raw["mean_predicted"], rc_raw["observed_frequency"], "o-", color="#c0392b",
                label="Uncalibrated", markersize=5)
        ax.plot(rc_cal["mean_predicted"], rc_cal["observed_frequency"], "s-", color="#27ae60",
                label=f"Calibrated ({final_calibrators[cand].name})", markersize=5)
        ax.set_title(cand, fontsize=12, fontweight="bold")
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Observed positive frequency")
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.legend(loc="upper left", fontsize=9)
        ax.grid(True, linestyle="--", alpha=0.4)
    plt.suptitle("Step 19: Calibration Curves on Validation Data (Uncalibrated vs Calibrated)",
                 fontsize=14, fontweight="bold", y=0.995)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_calibration_curves.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Brier score comparison (uncalibrated vs final calibrated, validation)
    uncal = [calibration_metrics(y_val_arr, val_probs[cand])["brier"] for cand in CANDIDATES]
    cal = [calibration_metrics(y_val_arr, final_calibrators[cand].transform(val_probs[cand]))["brier"]
           for cand in CANDIDATES]
    x = np.arange(len(CANDIDATES))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - width / 2, uncal, width, label="Uncalibrated", color="#c0392b")
    ax.bar(x + width / 2, cal, width, label="Calibrated", color="#27ae60")
    for xi, (u, c) in enumerate(zip(uncal, cal)):
        ax.text(xi - width / 2, u + 0.001, f"{u:.4f}", ha="center", fontsize=9)
        ax.text(xi + width / 2, c + 0.001, f"{c:.4f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(CANDIDATES, rotation=15, ha="right")
    ax.set_ylabel("Brier Score (lower = better)")
    ax.set_title("Step 19: Brier Score Comparison on Validation Data", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_brier_comparison.png", dpi=300, bbox_inches="tight")
    plt.close()


def generate_threshold_figure(
    sweep_df: pd.DataFrame,
    final_threshold: float,
) -> None:
    """Two-panel threshold trade-off visualization with the locked operating point."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

    ax1 = axes[0]
    ax1.plot(sweep_df["threshold"], sweep_df["recall_sensitivity"], "-", color="#439775",
             linewidth=2, label="Recall / Sensitivity")
    ax1.plot(sweep_df["threshold"], sweep_df["specificity"], "-", color="#6c5b7b",
             linewidth=2, label="Specificity")
    ax1.plot(sweep_df["threshold"], sweep_df["precision"], "--", color="#2b5c8f",
             linewidth=1.8, label="Precision")
    ax1.axvline(final_threshold, color="#c0392b", linestyle=":", linewidth=2,
                label=f"Locked threshold = {final_threshold:.2f}")
    ax1.axvline(0.5, color="gray", linestyle=":", linewidth=1.2, label="Default 0.50")
    ax1.set_xlabel("Decision Threshold", fontsize=11)
    ax1.set_ylabel("Metric Value", fontsize=11)
    ax1.set_title("Sensitivity / Specificity / Precision vs Threshold", fontsize=12, fontweight="bold")
    ax1.legend(loc="center left", fontsize=9)
    ax1.grid(True, linestyle="--", alpha=0.4)

    ax2 = axes[1]
    ax2.plot(sweep_df["threshold"], sweep_df["youden_j"], "-", color="#d65f5f",
             linewidth=2, label="Youden's J")
    ax2.plot(sweep_df["threshold"], sweep_df["balanced_accuracy"], "-", color="#2b5c8f",
             linewidth=2, label="Balanced Accuracy")
    ax2.plot(sweep_df["threshold"], sweep_df["f1"], "--", color="#e27c38",
             linewidth=1.8, label="F1 Score")
    ax2.axvline(final_threshold, color="#c0392b", linestyle=":", linewidth=2,
                label=f"Locked threshold = {final_threshold:.2f}")
    best_j = float(sweep_df["youden_j"].max())
    ax2.annotate(
        f"Max J = {best_j:.3f}",
        xy=(final_threshold, best_j),
        xytext=(final_threshold + 0.08, best_j - 0.06),
        arrowprops={"arrowstyle": "->", "color": "#c0392b"},
        fontsize=10,
        color="#c0392b",
        fontweight="bold",
    )
    ax2.set_xlabel("Decision Threshold", fontsize=11)
    ax2.set_ylabel("Metric Value", fontsize=11)
    ax2.set_title("Composite Selection Criteria vs Threshold", fontsize=12, fontweight="bold")
    ax2.legend(loc="lower center", fontsize=9)
    ax2.grid(True, linestyle="--", alpha=0.4)

    plt.suptitle("Step 20: Threshold Trade-off Analysis (Calibrated Validation Probabilities)",
                 fontsize=14, fontweight="bold", y=1.0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_threshold_tradeoff.png", dpi=300, bbox_inches="tight")
    plt.close()


def generate_test_figures(results: dict[str, Any]) -> None:
    """Final test figures: confusion matrix, ROC + PR curves, calibration curve."""
    y_test = results["y_test"]
    probs = results["probs"]
    cm = results["confusion_matrix"]
    threshold = results["threshold"]
    m = results["metrics"]

    # Panel 1: Confusion matrix heatmap
    fig, axes = plt.subplots(1, 3, figsize=(19, 6))
    group_names = ["TN", "FP", "FN", "TP"]
    group_counts = [f"{v}" for v in cm.flatten()]
    labels = [f"{name}\n{count}" for name, count in zip(group_names, group_counts)]
    labels = np.asarray(labels).reshape(2, 2)
    sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=True, square=True,
                linewidths=1.5, annot_kws={"size": 14, "fontweight": "bold"},
                xticklabels=["Predicted 0", "Predicted 1"],
                yticklabels=["Actual 0", "Actual 1"], ax=axes[0])
    axes[0].set_title(f"Confusion Matrix @ threshold {threshold:.2f}", fontsize=12, fontweight="bold")

    # Panel 2: ROC curve
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[1].plot(fpr, tpr, color="#2b5c8f", linewidth=2.2,
                 label=f"ROC-AUC = {m['roc_auc']:.4f} (95% CI [{m['roc_auc_ci95_low']:.3f}, {m['roc_auc_ci95_high']:.3f}])")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.6, linewidth=1, label="Chance")
    axes[1].set_xlabel("False Positive Rate", fontsize=11)
    axes[1].set_ylabel("True Positive Rate", fontsize=11)
    axes[1].set_title("Final Test ROC Curve (Locked Pipeline)", fontsize=12, fontweight="bold")
    axes[1].legend(loc="lower right", fontsize=9)
    axes[1].grid(True, linestyle="--", alpha=0.4)

    # Panel 3: Precision-Recall curve
    prec, rec, _ = precision_recall_curve(y_test, probs)
    axes[2].plot(rec, prec, color="#e27c38", linewidth=2.2,
                 label=f"PR-AUC = {m['pr_auc']:.4f} (95% CI [{m['pr_auc_ci95_low']:.3f}, {m['pr_auc_ci95_high']:.3f}])")
    axes[2].set_xlabel("Recall (Sensitivity)", fontsize=11)
    axes[2].set_ylabel("Precision", fontsize=11)
    axes[2].set_title("Final Test Precision-Recall Curve (Locked Pipeline)", fontsize=12, fontweight="bold")
    axes[2].legend(loc="lower left", fontsize=9)
    axes[2].grid(True, linestyle="--", alpha=0.4)

    plt.suptitle("Step 21: Final Locked Test Evaluation", fontsize=14, fontweight="bold", y=1.0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_test_confusion_roc_pr.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Test calibration curve (reliability diagram)
    rc = reliability_curve(y_test, probs)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.plot([0, 1], [0, 1], "k--", alpha=0.6, linewidth=1, label="Perfect calibration")
    ax.plot(rc["mean_predicted"], rc["observed_frequency"], "s-", color="#27ae60", markersize=6,
            label=f"Calibrated ({results['calibration_method']})")
    ax2 = ax.twinx()
    ax2.hist(probs, bins=20, alpha=0.25, color="#2b5c8f", label="Prediction density")
    ax2.set_ylabel("Count", fontsize=10)
    ax2.legend(loc="upper center", fontsize=9)
    ax.set_xlabel("Mean predicted probability (calibrated)", fontsize=11)
    ax.set_ylabel("Observed positive frequency", fontsize=11)
    ax.set_title(
        f"Final Test Calibration Curve — Brier = {m['brier_calibrated']:.4f}, ECE = {m['ece_calibrated']:.4f}",
        fontsize=12, fontweight="bold",
    )
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_test_calibration_curve.png", dpi=300, bbox_inches="tight")
    plt.close()


# -----------------------------------------------------------------------------
# Main Execution Pipeline
# -----------------------------------------------------------------------------
def main() -> None:
    start = time.time()
    print("=" * 80)
    print("PHASE 5: CALIBRATION, THRESHOLD ANALYSIS, FINAL LOCKED TEST EVALUATION")
    print("Steps 19-21 | Test split untouched until the final locked evaluation")
    print("=" * 80)

    # 1. Load data & locked Phase 4 configuration
    X_train, y_train, X_val, y_val = load_training_and_val_data()
    best_configs = p4.load_shortlisted_configs()
    with open(PHASE4_DIR / "ensemble_weights.json", "r", encoding="utf-8") as f:
        weights = json.load(f)
    y_train_arr = y_train.to_numpy()
    y_val_arr = y_val.to_numpy()
    print(f"\n[1/6] Data loaded: train n={len(X_train)} | val n={len(X_val)} | test n=LOCKED (not read)")
    print(f"  - Hybrid ensemble weights: { {k: round(v, 4) for k, v in weights.items()} }")

    # 2. Generate honest probability sets (OOF train + validation)
    print(f"\n[2/6] Generating OOF probabilities (Phase 4 protocol, seed {SEED}) + validation probabilities...")
    oof_probs, val_probs, ensemble, oof_ok = build_probability_sets(
        X_train, y_train, X_val, best_configs, weights
    )
    print(f"  - OOF reproduction matches Phase 4 saved probabilities: {oof_ok}")
    print(f"  - Candidates: {', '.join(CANDIDATES)}")

    # 3. Step 19 — Probability calibration
    print(f"\n[3/6] STEP 19: Probability Calibration (E07)...")
    _, comparison_df, _ = run_step19_calibration_part1(oof_probs, val_probs, y_train_arr, y_val_arr)
    step19 = run_step19_calibration_part2(val_probs, y_val_arr, comparison_df)
    final_calibrators = step19["final_calibrators"]
    generate_calibration_figures(val_probs, y_val_arr, final_calibrators)
    print("  [OK] fig_calibration_curves.png, fig_brier_comparison.png")

    # 4. Step 20 — Decision-threshold analysis (calibrated validation probabilities)
    print(f"\n[4/6] STEP 20: Decision-Threshold Analysis (E08)...")
    ensemble_calibrator = final_calibrators[ENSEMBLE_NAME]
    calibrated_val_probs = np.asarray(
        ensemble_calibrator.transform(val_probs[ENSEMBLE_NAME]), dtype=float
    )
    step20 = run_step20_threshold_analysis(calibrated_val_probs, y_val_arr)
    final_threshold = step20["final_threshold"]
    generate_threshold_figure(step20["sweep_df"], final_threshold)
    print("  [OK] fig_threshold_tradeoff.png")

    # 5. Step 21 — Final locked test evaluation (single authorized test read)
    print(f"\n[5/6] STEP 21: Final Locked Test Evaluation (E10)...")
    results = run_step21_locked_test_evaluation(
        ensemble, ensemble_calibrator, final_threshold, ensemble_calibrator.name,
        X_val, y_val_arr, calibrated_val_probs,
    )
    generate_test_figures(results)
    print("  [OK] fig_test_confusion_roc_pr.png, fig_test_calibration_curve.png")

    # 6. Final report & locked decision record
    print(f"\n[6/6] Final Test Metrics (LOCKED PIPELINE — these are the paper numbers):")
    for k, v in results["metrics"].items():
        print(f"  * {k:32s}: {v:.4f}")
    cm = results["confusion_matrix"]
    print(f"\n  Confusion Matrix @ threshold {final_threshold:.2f}:")
    print(f"    TN={cm[0, 0]:3d}  FP={cm[0, 1]:3d}")
    print(f"    FN={cm[1, 0]:3d}  TP={cm[1, 1]:3d}")

    decision_payload = {
        "phase": 5,
        "steps": [19, 20, 21],
        "experiment_ids": ["E07", "E08", "E10"],
        "primary_metric": "ROC-AUC",
        "step19_calibration": {
            "chosen_method_per_candidate": step19["chosen"],
            "final_procedure": {
                "candidate": ENSEMBLE_NAME,
                "calibrator": ensemble_calibrator.name,
                "fitted_on": "splits/val.csv",
            },
        },
        "step20_threshold": {
            "locked_threshold": final_threshold,
            "selection_rule": "Maximize Youden's J; ties resolved toward higher sensitivity",
            "validation_operating_point": {
                "recall_sensitivity": float(step20["best_row"]["recall_sensitivity"]),
                "specificity": float(step20["best_row"]["specificity"]),
                "f1": float(step20["best_row"]["f1"]),
            },
        },
        "step21_final_test": results["metrics"],
        "production_artifact": {
            "path": str(results["artifact_path"]),
            "contents": "Phase 4 hybrid ensemble + calibration + locked threshold",
        },
        "integrity": {
            "oof_reproduction_matches_phase4": oof_ok,
            "calibration_fitted_without_test_data": True,
            "threshold_selected_without_test_data": True,
            "test_split_accessed_exactly_once": _TEST_SPLIT_ACCESSED,
        },
        "decision": (
            "LOCKED. Final pipeline (ensemble + calibration + threshold) evaluated exactly once on "
            "the untouched test set. Results are final; no model, feature, calibration, or threshold "
            "changes are permitted without reopening the evaluation protocol."
        ),
    }
    with open(OUTPUT_DIR / "phase5_decision.json", "w", encoding="utf-8") as f:
        json.dump(decision_payload, f, indent=2)

    elapsed = time.time() - start
    print(f"\nPhase 5 (Steps 19-21) completed successfully in {elapsed:.1f}s!")
    print(f"All outputs and figures stored in: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()















Writing phase5_calibration_final_eval.py


In [7]:
%%writefile phase6_explainability_robustness.py
'''Phase 6 (Steps 22-23): explainability, error analysis, and subgroup robustness.

Uses the exact Phase 5 production artifact and locked threshold. The test set is
read only for analysis after Phase 5 has locked it; no model, calibration, or
threshold is refit here. SHAP is used when installed. A deterministic,
model-agnostic Shapley permutation fallback keeps the phase runnable in minimal
research environments where the optional ``shap`` package is unavailable.
'''
from __future__ import annotations
import json
import sys
import warnings
from pathlib import Path
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")
ROOT = Path(__file__).resolve().parent
OUT = ROOT / "models" / "phase6"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
BOOTSTRAPS = 400
MIN_GROUP = 10
# Importing this module registers the classes needed by artifacts produced when
# Phase 5 was executed as a script (pickle module name __main__).
sys.path.insert(0, str(ROOT))
import phase5_calibration_final_eval as p5  # noqa: E402

def load_locked_data() -> tuple[pd.DataFrame, pd.Series, Any, float]:
    manifest = json.loads((ROOT / "models" / "phase5" / "lock_manifest.json").read_text(encoding="utf-8"))
    artifact_path = Path(manifest["artifact_path"])
    if not artifact_path.is_absolute():
        artifact_path = ROOT / artifact_path
    artifact = p5.load_production_artifact(artifact_path)
    test = pd.read_csv(ROOT / "splits" / "test.csv")
    if "target" not in test or len(test) == 0:
        raise ValueError("splits/test.csv must contain a non-empty target column")
    X = test.drop(columns=["target"])
    y = test["target"].astype(int)
    return X, y, artifact, float(manifest["locked_pipeline"]["decision_threshold"])


def metrics(y: pd.Series | np.ndarray, probs: np.ndarray, threshold: float) -> dict[str, float]:
    yv = np.asarray(y, dtype=int)
    pred = (probs >= threshold).astype(int)
    tn = int(((yv == 0) & (pred == 0)).sum())
    fp = int(((yv == 0) & (pred == 1)).sum())
    return {
        "n": int(len(yv)), "positive_rate": float(yv.mean()),
        "roc_auc": float(roc_auc_score(yv, probs)) if len(np.unique(yv)) == 2 else np.nan,
        "pr_auc": float(average_precision_score(yv, probs)) if len(np.unique(yv)) == 2 else np.nan,
        "recall_sensitivity": float(recall_score(yv, pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "precision": float(precision_score(yv, pred, zero_division=0)),
        "f1": float(f1_score(yv, pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(yv, pred)),
    }


def bootstrap_ci(y: np.ndarray, probs: np.ndarray, threshold: float, seed: int = SEED) -> dict[str, float]:
    rng = np.random.default_rng(seed)
    values: dict[str, list[float]] = {"roc_auc": [], "recall_sensitivity": [], "specificity": [], "f1": []}
    for _ in range(BOOTSTRAPS):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        row = metrics(y[idx], probs[idx], threshold)
        for key in values:
            values[key].append(row[key])
    result: dict[str, float] = {}
    for key, vals in values.items():
        result[f"{key}_ci95_low"] = float(np.percentile(vals, 2.5)) if vals else np.nan
        result[f"{key}_ci95_high"] = float(np.percentile(vals, 97.5)) if vals else np.nan
    return result

def predict_fn(artifact: Any, columns: list[str]):
    def fn(values: np.ndarray) -> np.ndarray:
        frame = pd.DataFrame(values, columns=columns)
        return np.asarray(artifact.predict_proba(frame)[:, 1], dtype=float)
    return fn

def fallback_shap(artifact: Any, X: pd.DataFrame, n_rows: int = 8) -> tuple[np.ndarray, str]:
    '''Deterministic model-agnostic Shapley permutation approximation.'''
    rng = np.random.default_rng(SEED)
    columns = list(X.columns)
    sample = X.iloc[: min(n_rows, len(X))].copy()
    background = X.iloc[: min(40, len(X))].copy()
    baseline = background.mode(dropna=False).iloc[0]
    phi = np.zeros((len(sample), len(columns)), dtype=float)
    fn = predict_fn(artifact, columns)
    for row_i, (_, row) in enumerate(sample.iterrows()):
        for _ in range(4):
            order = rng.permutation(len(columns))
            current = pd.DataFrame([baseline.to_dict()])
            previous = float(fn(current.to_numpy())[0])
            for j in order:
                current.iloc[0, j] = row.iloc[j]
                updated = float(fn(current.to_numpy())[0])
                phi[row_i, j] += updated - previous
                previous = updated
    phi /= 4.0
    return phi, "deterministic_model_agnostic_shapley_permutation_fallback"


def explain_with_shap(artifact: Any, X: pd.DataFrame) -> tuple[np.ndarray, str]:
    columns = list(X.columns)
    try:
        import shap  # type: ignore
        background = X.iloc[: min(40, len(X))]
        sample = X.iloc[: min(30, len(X))]
        explainer = shap.KernelExplainer(predict_fn(artifact, columns), background.to_numpy(), seed=SEED)
        values = explainer.shap_values(sample.to_numpy(), nsamples=80)
        if isinstance(values, list):
            values = values[-1]
        return np.asarray(values, dtype=float), "shap.KernelExplainer"
    except Exception as exc:
        print(f"[warning] Native SHAP unavailable; using deterministic fallback: {exc}")
        return fallback_shap(artifact, X)


def run() -> None:
    X, y, artifact, threshold = load_locked_data()
    probs = np.asarray(artifact.predict_proba(X)[:, 1], dtype=float)
    pred = (probs >= threshold).astype(int)
    base = metrics(y, probs, threshold)
    base.update(bootstrap_ci(y.to_numpy(), probs, threshold))

    # Step 22: transparent global/local explanations and mistakes.
    phi, explanation_method = explain_with_shap(artifact, X)
    explain_X = X.iloc[: len(phi)].reset_index(drop=True)
    global_df = pd.DataFrame({"feature": X.columns, "mean_abs_shap": np.mean(np.abs(phi), axis=0), "mean_shap": np.mean(phi, axis=0)})
    global_df["rank"] = global_df["mean_abs_shap"].rank(method="min", ascending=False).astype(int)
    global_df.sort_values("mean_abs_shap", ascending=False).to_csv(OUT / "shap_global_importance.csv", index=False)
    local = explain_X.copy()
    for i, col in enumerate(X.columns):
        local[f"shap_{col}"] = phi[:, i]
    local["row_index"] = explain_X.index
    local["actual"] = y.iloc[: len(phi)].to_numpy()
    local["predicted"] = pred[: len(phi)]
    local["probability"] = probs[: len(phi)]
    local.to_csv(OUT / "shap_local_explanations.csv", index=False)
    plt.figure(figsize=(9, 6))
    top = global_df.sort_values("mean_abs_shap").tail(14)
    plt.barh(top["feature"], top["mean_abs_shap"], color="#2b5c8f")
    plt.xlabel("Mean absolute SHAP value (model output scale)")
    plt.title(f"Phase 6 global explainability ({explanation_method})")
    plt.tight_layout(); plt.savefig(OUT / "fig_shap_global_importance.png", dpi=220); plt.close()

    errors = X.copy()
    errors.insert(0, "row_index", X.index)
    errors["actual"] = y.to_numpy(); errors["predicted"] = pred; errors["probability"] = probs
    errors["error_type"] = np.select([(y.to_numpy() == 0) & (pred == 1), (y.to_numpy() == 1) & (pred == 0)], ["false_positive", "false_negative"], default="correct")
    errors["confidence"] = np.maximum(probs, 1.0 - probs)
    errors.to_csv(OUT / "error_analysis_all_predictions.csv", index=False)
    errors[errors["error_type"] != "correct"].sort_values("confidence", ascending=False).to_csv(OUT / "error_analysis_mistakes.csv", index=False)
    high_conf = errors[(errors["error_type"] != "correct") & (errors["confidence"] >= 0.80)]
    high_conf.to_csv(OUT / "error_analysis_high_confidence_mistakes.csv", index=False)
    summary = errors.groupby("error_type", as_index=False).agg(count=("row_index", "size"), mean_confidence=("confidence", "mean"), mean_probability=("probability", "mean"))
    summary.to_csv(OUT / "error_analysis_summary.csv", index=False)

    # Step 23: predefined, non-tuned subgroup analysis plus uncertainty.
    groups: dict[str, pd.Series] = {
        "sex=0": X["sex"] == 0, "sex=1": X["sex"] == 1,
        "cp=0": X["cp"] == 0, "cp=1": X["cp"] == 1, "cp=2": X["cp"] == 2, "cp=3": X["cp"] == 3,
        "exang=0": X["exang"] == 0, "exang=1": X["exang"] == 1,
        "age<55": X["age"] < 55, "age=55-64": X["age"].between(55, 64), "age>=65": X["age"] >= 65,
    }
    rows: list[dict[str, Any]] = []
    for name, mask in groups.items():
        idx = mask.to_numpy()
        if int(idx.sum()) < MIN_GROUP or len(np.unique(y.to_numpy()[idx])) < 2:
            continue
        row = {"subgroup": name, **metrics(y.to_numpy()[idx], probs[idx], threshold), **bootstrap_ci(y.to_numpy()[idx], probs[idx], threshold, SEED + len(rows))}
        rows.append(row)
    subgroup_df = pd.DataFrame(rows)
    subgroup_df.to_csv(OUT / "subgroup_metrics_with_bootstrap_ci.csv", index=False)
    threshold_rows = []
    for t in [0.45, 0.50, 0.51, 0.55, 0.60]:
        threshold_rows.append({"threshold": t, **metrics(y, probs, t)})
    pd.DataFrame(threshold_rows).to_csv(OUT / "threshold_sensitivity_robustness.csv", index=False)

    decision = {"phase": 6, "steps": [22, 23], "locked_threshold": threshold, "explanation_method": explanation_method, "base_test_metrics": base, "error_counts": summary.to_dict(orient="records"), "subgroups_evaluated": len(rows), "minimum_subgroup_n": MIN_GROUP, "limitations": ["Subgroups are exploratory and not fairness or clinical validation.", "Small sample sizes produce wide uncertainty; groups below the minimum size or without both classes are omitted.", "The test set was used only after Phase 5 lock for descriptive post-hoc analysis; no fitting or tuning was performed."]}
    (OUT / "phase6_decision.json").write_text(json.dumps(decision, indent=2, default=str), encoding="utf-8")
    print(f"Phase 6 completed successfully: {len(X)} test rows, {len(rows)} subgroups, method={explanation_method}")

if __name__ == "__main__":
    run()


Writing phase6_explainability_robustness.py


In [8]:
%%writefile phase7_production_handoff.py
'''Phase 7 (Steps 24-27): production handoff, validation, cleanup, and report.

The Phase 5 artifact remains authoritative. This phase only packages and checks
that artifact; it never retrains, calibrates, tunes, or changes the threshold.
'''
from __future__ import annotations
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))
import phase5_calibration_final_eval as p5  # noqa: E402
OUT = ROOT / "models" / "phase7"
ARTIFACT = ROOT / "models" / "phase5" / "hybrid_ensemble_calibrated_production.pkl"
MANIFEST = ROOT / "models" / "phase5" / "lock_manifest.json"
FEATURE_COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"]


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def check(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)


def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    check(ARTIFACT.exists(), f"Missing locked artifact: {ARTIFACT}")
    check(MANIFEST.exists(), f"Missing lock manifest: {MANIFEST}")
    manifest: dict[str, Any] = json.loads(MANIFEST.read_text(encoding="utf-8"))
    actual_hash = sha256(ARTIFACT)
    expected_hash = manifest.get("artifact_hash_sha256")
    check(actual_hash == expected_hash, "Artifact SHA-256 does not match Phase 5 lock manifest")
    model = p5.load_production_artifact(ARTIFACT)
    threshold = float(manifest["locked_pipeline"]["decision_threshold"])
    check(abs(float(model.threshold) - threshold) < 1e-12, "Artifact threshold differs from locked threshold")

    train = pd.read_csv(ROOT / "splits" / "train.csv")
    test = pd.read_csv(ROOT / "splits" / "test.csv")
    check(list(train.drop(columns=["target"]).columns) == FEATURE_COLUMNS, "Training schema changed")
    check(list(test.drop(columns=["target"]).columns) == FEATURE_COLUMNS, "Test schema changed")
    X_test = test[FEATURE_COLUMNS]
    y_test = test["target"].astype(int).to_numpy()
    probabilities = np.asarray(model.predict_proba(X_test)[:, 1], dtype=float)
    predictions = np.asarray(model.predict(X_test), dtype=int)
    check(len(probabilities) == len(test), "Inference row count mismatch")
    check(bool(np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()), "Invalid probabilities")
    check(bool(np.array_equal(predictions, (probabilities >= threshold).astype(int))), "Prediction does not use locked threshold")
    roundtrip = p5.load_production_artifact(ARTIFACT)
    roundtrip_probs = np.asarray(roundtrip.predict_proba(X_test)[:, 1], dtype=float)
    check(float(np.max(np.abs(probabilities - roundtrip_probs))) < 1e-12, "Artifact round-trip mismatch")

    validation = {
        "phase": 7, "steps": [24, 25], "status": "passed",
        "artifact": str(ARTIFACT.relative_to(ROOT)), "artifact_sha256": actual_hash,
        "locked_threshold": threshold, "feature_columns": FEATURE_COLUMNS,
        "test_rows_validated": int(len(test)), "probability_range": [float(probabilities.min()), float(probabilities.max())],
        "round_trip_max_probability_difference": float(np.max(np.abs(probabilities - roundtrip_probs))),
        "test_predictions_recomputed": True,
        "warning": "This validation reuses the already locked test split for inference regression only; no fitting or tuning was performed.",
    }
    (OUT / "inference_validation.json").write_text(json.dumps(validation, indent=2), encoding="utf-8")
    pd.DataFrame({"row_index": np.arange(len(test)), "probability": probabilities, "prediction": predictions, "actual": y_test}).to_csv(OUT / "inference_validation_predictions.csv", index=False)

    inventory = []
    for path in sorted(ROOT.rglob("*")):
        if path.is_file() and ".git" not in path.parts and "__pycache__" not in path.parts:
            inventory.append({"path": str(path.relative_to(ROOT)), "bytes": path.stat().st_size, "sha256": sha256(path) if path.suffix in {".py", ".json", ".md", ".pkl"} else None})
    (OUT / "repository_inventory.json").write_text(json.dumps({"phase": 7, "files": inventory}, indent=2), encoding="utf-8")

    metrics_path = ROOT / "models" / "phase5" / "final_test_metrics.csv"
    metrics = pd.read_csv(metrics_path).set_index("metric")["value"].to_dict() if metrics_path.exists() else {}
    phase6_path = ROOT / "models" / "phase6" / "phase6_decision.json"
    phase6 = json.loads(phase6_path.read_text(encoding="utf-8")) if phase6_path.exists() else {}
    report = {
        "title": "Heart Disease Classification — Phase 7 Production Handoff",
        "generated_utc": datetime.now(timezone.utc).isoformat(), "status": "complete",
        "steps": {"24_production_artifact": "passed", "25_inference_validation": "passed", "26_repository_cleanup": "passed", "27_final_report": "passed"},
        "production_contract": {"artifact": str(ARTIFACT.relative_to(ROOT)), "calibration": manifest["locked_pipeline"]["calibration"], "threshold": threshold, "schema": FEATURE_COLUMNS},
        "locked_test_metrics_from_phase5": metrics, "phase6_summary": {"explanation_method": phase6.get("explanation_method"), "subgroups_evaluated": phase6.get("subgroups_evaluated")},
        "reproducibility": {"python": sys.version, "platform": platform.platform(), "artifact_sha256": actual_hash},
        "limitations": ["Small single-cohort dataset; no external or temporal validation.", "Test performance is an evaluation result, not a clinical performance guarantee.", "Research and educational use only; not a diagnostic device."],
    }
    (OUT / "final_report.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
    (OUT / "FINAL_REPORT.md").write_text(render_report(report), encoding="utf-8")
    print(f"Phase 7 completed successfully: artifact verified, {len(test)} inference rows validated, report written to {OUT}")


def render_report(report: dict[str, Any]) -> str:
    metrics = report["locked_test_metrics_from_phase5"]
    lines = ["# Phase 7 — Production Handoff Report", "", "## Status", "Complete: Steps 24–27 passed.", "", "## Production contract", f"- Artifact: `{report['production_contract']['artifact']}`", f"- Locked threshold: `{report['production_contract']['threshold']}`", f"- Calibration: `{report['production_contract']['calibration'].get('method', 'unknown')}`", f"- SHA-256: `{report['reproducibility']['artifact_sha256']}`", "", "## Locked Phase 5 test metrics"]
    lines.extend(f"- {key}: {value}" for key, value in metrics.items())
    lines.extend(["", "## Validation", "- Input schema, probability range, threshold behavior, serialization round-trip, and row counts passed.", "- No fitting, calibration, tuning, or threshold selection was performed in Phase 7.", "", "## Limitations", *[f"- {item}" for item in report["limitations"]]])
    return "\n".join(lines) + "\n"

if __name__ == "__main__":
    main()


Writing phase7_production_handoff.py


In [9]:
%%writefile production_inference.py
'''Stable inference API for the locked Phase 5 heart-disease artifact.

This module performs input validation before calling the serialized production
pipeline. It deliberately does not fit, calibrate, tune, or alter the model.
Research use only; not a clinical diagnostic device.
'''
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Mapping
import numpy as np
import pandas as pd
ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))
import phase5_calibration_final_eval as p5  # noqa: E402
MANIFEST_PATH = ROOT / "models" / "phase5" / "lock_manifest.json"
FEATURE_COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]


def _resolve_artifact_path(manifest: dict[str, Any]) -> Path:
    raw = Path(str(manifest["artifact_path"]))
    return raw if raw.is_absolute() and raw.exists() else ROOT / "models" / "phase5" / raw.name

def load_model() -> tuple[Any, dict[str, Any]]:
    '''Load the exact locked artifact and manifest; never retrain.'''
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    artifact_path = _resolve_artifact_path(manifest)
    if not artifact_path.exists():
        raise FileNotFoundError(f"Locked production artifact not found: {artifact_path}")
    return p5.load_production_artifact(artifact_path), manifest

def validate_input(values: Mapping[str, Any] | pd.DataFrame) -> pd.DataFrame:
    '''Validate and normalize one row or a dataframe with the production schema.'''
    frame = values.copy() if isinstance(values, pd.DataFrame) else pd.DataFrame([dict(values)])
    missing = [column for column in FEATURE_COLUMNS if column not in frame.columns]
    extra = [column for column in frame.columns if column not in FEATURE_COLUMNS]
    if missing or extra:
        raise ValueError(f"Invalid feature schema; missing={missing}, extra={extra}")
    frame = frame.loc[:, FEATURE_COLUMNS].copy()
    for column in FEATURE_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    if frame.isna().any().any() or not np.isfinite(frame.to_numpy(dtype=float)).all():
        raise ValueError("Input contains missing or non-finite feature values")
    if len(frame) == 0:
        raise ValueError("At least one input row is required")
    return frame

def predict(values: Mapping[str, Any] | pd.DataFrame) -> pd.DataFrame:
    '''Return calibrated probability and locked-threshold decision for each row.'''
    model, manifest = load_model()
    frame = validate_input(values)
    probabilities = np.asarray(model.predict_proba(frame)[:, 1], dtype=float)
    decisions = np.asarray(model.predict(frame), dtype=int)
    threshold = float(manifest["locked_pipeline"]["decision_threshold"])
    return pd.DataFrame({"probability": probabilities, "prediction": decisions, "threshold": threshold})


def predict_one(values: Mapping[str, Any]) -> dict[str, Any]:
    '''Predict one validated feature mapping and return JSON-friendly values.'''
    if not isinstance(values, Mapping):
        raise TypeError("predict_one expects a mapping of feature names to values")
    row = predict(values).iloc[0].to_dict()
    result: dict[str, Any] = {}
    for key, value in row.items():
        result[str(key)] = value.item() if hasattr(value, "item") else value
    return result


Writing production_inference.py


## Execute Pipeline
The following cells will execute each phase of the pipeline sequentially.

In [10]:
!python phase1_foundation.py

Step 1: Loading data...
Data shape: (1025, 14)

Step 2: Data audit...

Columns and data types:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64
dtype: object

Missing values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Duplicate rows: 723

Basic statistics:
               age          sex  ...         thal       target
count  1025.000000  1025.000000  ...  1025.000000  1025.000000
mean     54.434146     0.695610  ...     2.323902     0.513171
std       9.072290     0.460373  ...     0.620660     0.500070
min      29.000000     0.000000  ...     0.000000     0.000000
25%      48.000000     0.00000

In [11]:
!python phase2_modeling_foundation.py

Phase 2: Modeling foundation

1. Loading train, validation, and test splits...
Train shape: (615, 14)
Validation shape: (205, 14)
Test shape: (205, 14)

Feature shapes:
X_train: (615, 13), y_train: (615,)
X_val: (205, 13), y_val: (205,)
X_test: (205, 13), y_test: (205,)

Numeric features (6): ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
Categorical features (7): ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

2. Fitting preprocessor on training data...
Processed feature shapes:
X_train_processed: (615, 26)
X_val_processed: (205, 26)
X_test_processed: (205, 26)

3. Skipping polynomial feature engineering for baseline models.

3.5. Saving preprocessor and processed data...
Preprocessor saved to models/preprocessor.pkl
Processed data saved to splits/ directory.

4. Training baseline models...

Training Logistic Regression...
Validation Accuracy: 0.8585
Validation Precision: 0.8654
Validation Recall: 0.8571
Validation F1: 0.8612
Validation ROC-AUC: 0.9040
Model saved 

In [12]:
!python phase3_advanced.py

Phase 3: advanced modeling (steps 13-16)
Training rows: 615; features: 13; positive rate: 0.514

Phase 3 completed successfully.
              model  best_cv_roc_auc  roc_auc_mean  roc_auc_std  selected_for_phase4
      Random Forest         0.990051      0.991076     0.001493                 True
                KNN         0.991772      0.988683     0.002873                 True
                SVM         0.978404      0.977283     0.004479                 True
Logistic Regression         0.926014      0.925263     0.001131                False
Outputs: /content/models/phase3


In [13]:
!python phase4_ensemble_ablation.py

PHASE 4: RESEARCH CENTERPIECE - HYBRID ENSEMBLE & ABLATION STUDY
Steps 17 & 18 | Strict Anti-Leakage Protocol (Test Set Locked)

[1/7] Data Loaded:
  - Training samples: 615 (Positive: 316, Negative: 299)
  - Validation samples: 205 (Holdout checkpoint only)
  - Test partition: STRICTLY LOCKED and NOT ACCESSED

[2/7] Phase 3 Shortlisted Configurations Loaded:
  * KNN           : Strategy = smote        | Params = {'model__n_neighbors': 11, 'model__weights': 'distance'}
  * SVM           : Strategy = class_weight | Params = {'model__C': 10.0, 'model__gamma': 'scale'}
  * Random Forest : Strategy = class_weight | Params = {'model__max_depth': 10, 'model__max_features': 'log2', 'model__min_samples_leaf': 1}

[3/7] Generating 5-Fold Stratified OOF Probabilities on Training Data...

[4/7] Step 17: Hybrid Ensemble Weight Optimization...
  - Optimal Weights (Minimizing OOF Brier Score):
    * KNN           : 0.8479 (84.8%)
    * SVM           : 0.0000 (0.0%)
    * Random Forest : 0.1521 (15.2

In [14]:
!python phase5_calibration_final_eval.py

PHASE 5: CALIBRATION, THRESHOLD ANALYSIS, FINAL LOCKED TEST EVALUATION
Steps 19-21 | Test split untouched until the final locked evaluation

[1/6] Data loaded: train n=615 | val n=205 | test n=LOCKED (not read)
  - Hybrid ensemble weights: {'KNN': 0.8479, 'SVM': 0.0, 'Random Forest': 0.1521}

[2/6] Generating OOF probabilities (Phase 4 protocol, seed 42) + validation probabilities...
  [integrity] OOF reproduction KNN: max|diff| = 1.11e-16
  [integrity] OOF reproduction SVM: max|diff| = 1.11e-16
  [integrity] OOF reproduction Random Forest: max|diff| = 2.22e-16
  - OOF reproduction matches Phase 4 saved probabilities: True
  - Candidates: KNN, SVM, Random Forest, Hybrid Ensemble (Weighted)

[3/6] STEP 19: Probability Calibration (E07)...

--- Uncalibrated Calibration Assessment (Brier / ECE / LogLoss) ---
                 candidate      split calibration    brier      ece  log_loss
                       KNN  train_oof        none 0.029388 0.022817  0.121975
                       KNN 

In [15]:
!python phase6_explainability_robustness.py

100% 30/30 [00:09<00:00,  3.30it/s]
Phase 6 completed successfully: 205 test rows, 11 subgroups, method=shap.KernelExplainer


In [16]:
!python phase7_production_handoff.py

Phase 7 completed successfully: artifact verified, 205 inference rows validated, report written to /content/models/phase7
